# Power BI Wide-Format Forecast Export

Task 4 (last) of the weekly ARR forecasting job: runs after `01_ucc_forecast_foundation` / `02_itsg_growth_arr_forecasting` (parallel) and `03_forecast_unify_publish`. See `notebooks/README.md`.

Reads the already-calibrated predictions from `forecast_pure_predictions` (built by notebook 03), pivots them to wide format, applies the BOQ plan override for UCC forward weeks, and writes `arr_forecast_powerbi_wide` + `arr_forecast_long` (+ run history), then syncs the app tables to `federated.sales`.

## Output tables (total grain only, as of 2026-10-08)
* `datagroup_mdl.mdl_sales_analytics.arr_forecast_powerbi_wide` - wide, **Total / UCC / ITSG rows only** (`sales_market = 'Total'`, `grain_level = 'total'`; about 92 weeks from 2026-01 to ~2027-10 = 276 rows). No region / market rows are published; anything that filters this table by a market other than `Total` now gets no rows.
* `datagroup_mdl.mdl_sales_analytics.arr_forecast_long` - long (unpivoted) fact table, same total-only scope
* `datagroup_mdl.mdl_sales_analytics.arr_forecast_wide_history` - one appended snapshot per run date (deduplicated per day)
* Federated copy: `federated.sales.*` (14 tables synced in the last cell; `forecast_app_snapshot*` is written by notebook 03)

Why total-only: market rows were allocated shares of the total, had no Static/Dynamic values on closed weeks, and for ITSG did not add back up to the total (differences of $1.6K-$12.8K per week). The filter is the "TOTAL GRAIN ONLY" cell, before the long table and every write. Intermediate row counts printed by earlier cells (e.g. 1,680 or 2,205 rows) still include market rows; the published count is the one printed by that cell.

## Data flow
1. **Read `forecast_pure_predictions`** (Total grain, 6 models, real p05/p25/p50/p75/p95) and rename models (`Global_LGB_Q50_UCC`/`_ITSG` -> `Global_LGB_Q50`, `Prophet_trend` -> `Prophet`).
2. **Pivot** to `{Model}_P05 ... {Model}_P95` columns (identity map; legacy p10/p30/p70/p90 removed 2026-09-28).
3. **Scenarios** from `Adaptive_Ensemble`: `most_likely` = blended P50 (actual where the week is closed), `worst_case` = P25, `best_case` = P75.
4. **Forward `most_likely` / `blended_p50` override (C1b):**
   * **UCC** - taken from the frozen BOQ plan, `ucc_forecast_boq.forecast`, for the **current fiscal year only** (through Dec 31 of the run year), so the export matches `forecast_app_snapshot`. This BOQ value already includes UCC's quarter-end factor and the +$200K last-week-of-quarter add (notebook 01).
   * **ITSG** - taken from the `itsg_forecast_cbe` forecast column (BOQ-paced). ITSG is **not** switched to its BOQ table.
   * **Total** = UCC + ITSG.
5. **Closed-week gap fill** - weeks missing from `forecast_pure_predictions` are filled from the CBE actuals tables; model quantile columns on those rows are backfilled from the vintage tables, whose history only has p10/p50/p90, so p05/p25/p75/p95 on those backfilled rows are approximations.
6. **Static / Dynamic pair (Total grain)** - `Static_P50` = `Adaptive_Ensemble_P50` (the model's original call, never overwritten, including closed weeks); `Dynamic_P50` = `blended_p50` / `most_likely` (actual once closed, otherwise the forward value after the step-4 override).
7. **Market-level actuals** are still read from `forecast_all_models` and market forecasts are still built in the intermediate cells, but all market rows are dropped by the total-only filter before anything is written.
8. **Long-format table** - unpivoted fact table joined to per-model holdout WAPE from the monitoring baseline where available.

## Period-end uplift: diagnostic only (NOT applied)
The three-tier (year-end / quarter-end / month-end) uplift cell only **prints** what each factor would be. It is not multiplied into any output column (UPLIFT_* = 1.0). Reason: the lookback filter (years strictly before the current year) can never match this table's ~2026-onward scope, and UCC's own quarter-end calibration (notebook 01, Step 14) already corrects quarter-end upstream, so a second live multiplier would double-count. Re-enable only as a deliberate decision to retire or coordinate with notebook 01's calibration.

## Not included in this export (dashboard / app differences to be aware of)
* **Miradore** - not in this table. The app's "Include Miradore" view reads `forecast_app_snapshot_incl_miradore` (notebook 03).
* **+$200K SelfServeAdjustment** - the UCC BOQ value above already contains it. A dashboard measure that adds another $200K on the last quarter week would count it twice.
* `mc_actuals` and `miradore_plan` are market-level / monthly-broadcast sources and are normally empty at total grain; do not rely on them.

## Models (6)
Adaptive_Ensemble, DHR_ARIMA, ETS, Global_LGB_Q50, MSTL_v2, Prophet

## Column groups

| Group | Columns | Description |
|-------|---------|-------------|
| **Dimensions** | `ds`, `product`, `sales_market`, `grain_level`, `forecast_type` | Row identity. `forecast_type` is consistent per date: `actuals` for closed weeks, `forecast` for future weeks |
| **Calendar** | `year`, `ISOWeek`, `quarter`, `month`, `RelWeekNumber`, `WeekLabel`, `Product_group` | Derived time + grouping |
| **Model quantiles** | `{Model}_P05` ... `{Model}_P95` (6 models x 5 = 30 cols) | Native, upstream-calibrated quantiles from `forecast_pure_predictions` |
| **Shared** | `actuals`, `mc_actuals`, `miradore_plan` | Actual ARR; the last two are normally empty at total grain (see above) |
| **Scenarios** | `most_likely`, `best_case`, `worst_case`, `blended_p50` | Adaptive Ensemble P50 / P75 / P25 with the UCC BOQ override applied; no uplift |
| **Static / Dynamic** | `Static_P50`, `Dynamic_P50` | Frozen model call vs current best estimate (see step 6) |
| **YTD** | `ARR_YTD` | Running sum of actuals by product x year; NULL past the last actual |
| **Metadata** | `run_date`, `published_at_utc` | Lineage (single snapshot per run) |
| **Quality flags** | `model_fill_source` | Provenance: `cbe_actuals`, `historical`, or the source label from `forecast_pure_predictions` |

## Known limitations
* **Single snapshot** - `run_date` is constant per run. Use `arr_forecast_wide_history` for vintage comparison.
* **Calibration and reconciliation live upstream** - this notebook does no conformal recalibration or reconciliation (the old V1 logic sits in `if False:` blocks of the no-op cells for reference only). Validate coverage / APE figures in notebooks 01-03, not here.
* **Printed "Holdout evaluation period starts" date** is a fixed value (2026-01-05) and is not read from the UCC/ITSG notebooks; treat it as informational.
* **Expected harmless messages** - `GRANT ... federated.sales` is skipped with PERMISSION_DENIED (no MANAGE on the `federated` catalog), and an "ITSG market override skipped" log line comes from the discarded market-level path.
* **Forward weeks beyond the current fiscal year** for UCC are not BOQ-overridden; they use the model P50 from `forecast_pure_predictions`.


In [ ]:
%sql
GRANT SELECT ON TABLE datagroup_mdl.mdl_sales_analytics.forecast_app_snapshot 
TO `324a6ec7-e988-42c7-8a7f-55465f5bea37`

In [ ]:
%sql --name pbi_monthly
SELECT
  product,
  DATE_FORMAT(ds, 'yyyy-MM') AS month,
  CONCAT('Q', quarter) AS qtr,
  ROUND(SUM(actuals) / 1e6, 2)                   AS `Actuals ($M)`,
  ROUND(SUM(Adaptive_Ensemble_P50) / 1e6, 2)      AS `Ensemble ($M)`,
  ROUND(SUM(Prophet_P50) / 1e6, 2)                 AS `Prophet ($M)`,
  ROUND(SUM(MSTL_v2_P50) / 1e6, 2)                AS `MSTL ($M)`,
  ROUND(SUM(DHR_ARIMA_P50) / 1e6, 2)              AS `DHR ($M)`,
  ROUND(SUM(Global_LGB_Q50_P50) / 1e6, 2)         AS `LGB ($M)`,
  ROUND(SUM(ETS_P50) / 1e6, 2)                    AS `ETS ($M)`,
  ROUND(SUM(most_likely) / 1e6, 2)                AS `Most Likely ($M)`,
  ROUND(SUM(worst_case) / 1e6, 2)                 AS `Worst ($M)`,
  ROUND(SUM(best_case) / 1e6, 2)                  AS `Best ($M)`,
  COUNT(DISTINCT CASE WHEN actuals IS NOT NULL THEN ds END) AS `Act Wks`,
  COUNT(DISTINCT CASE WHEN forecast_type = 'forecast' THEN ds END) AS `Fwd Wks`,
  FIRST(model_fill_source) AS fill_source
FROM datagroup_mdl.mdl_sales_analytics.arr_forecast_powerbi_wide
WHERE UPPER(TRIM(sales_market)) = 'TOTAL'
  AND grain_level = 'total'
  AND ds >= '2026-01-01' AND ds <= '2026-12-28'
GROUP BY product, DATE_FORMAT(ds, 'yyyy-MM'), quarter
ORDER BY product, month

In [ ]:
# ============================================================================
# CONFIG  (V2 — sourced from forecast_pure_predictions)
# ============================================================================
# CHANGED vs V1:
#   - Source: forecast_pure_predictions (calibrated bands, all models unified)
#   - Quantiles: Output columns use P05/P25/P50/P75/P95, real native values from
#     forecast_pure_predictions (legacy p10/p30/p70/p90 mapping removed 2026-09-28)
#   - No hindcast backfill needed (already in static table)
#   - No conformal recalibration needed (already calibrated)
#   - Market-level actuals from forecast_all_models (forecasts suppressed)
# ============================================================================

CATALOG = "datagroup_mdl"
SCHEMA  = "mdl_sales_analytics"

PURE_TABLE       = f"{CATALOG}.{SCHEMA}.forecast_pure_predictions"
MARKET_SOURCE    = f"{CATALOG}.{SCHEMA}.forecast_all_models"
OUTPUT_TABLE     = f"{CATALOG}.{SCHEMA}.arr_forecast_powerbi_wide"
LONG_TABLE       = f"{CATALOG}.{SCHEMA}.arr_forecast_long"

APP_SERVICE_PRINCIPAL = "324a6ec7-e988-42c7-8a7f-55465f5bea37"

# Normalized model names for PowerBI columns (LGB merged, Prophet_trend → Prophet)
MODELS = [
    "Adaptive_Ensemble",
    "DHR_ARIMA",
    "ETS",
    "Global_LGB_Q50",
    "MSTL_v2",
    "Prophet",
]

# 2026-09-28: forecast_pure_predictions now publishes real, native
# p05/p25/p50/p75/p95 (legacy p10/p20/.../p90 grid removed everywhere in the
# unify_publish notebook), so this is a straight identity map. Kept as a dict
# (rather than inlining the column names) because the vintages fallback
# section further down produces its own p05/p25/p75/p95 columns via
# interpolation/approximation from the vintages table's native p10/p50/p90 —
# reusing this same map keeps both pivots consistent.
QUANTILES = ["P05", "P25", "P50", "P75", "P95"]
QUANTILE_MAP = {
    "P05": "p05", "P25": "p25", "P50": "p50", "P75": "p75", "P95": "p95",
}

# Model name normalization (static table → PowerBI column name)
MODEL_RENAME = {
    'Global_LGB_Q50_UCC':  'Global_LGB_Q50',
    'Global_LGB_Q50_ITSG': 'Global_LGB_Q50',
    'Prophet_trend':       'Prophet',
}

print(f"Source: {PURE_TABLE}")
print(f"Market actuals: {MARKET_SOURCE}")
print(f"Output: {OUTPUT_TABLE}")
print(f"Models: {len(MODELS)}, Quantiles: {len(QUANTILES)}")
print(f"Expected model columns: {len(MODELS) * len(QUANTILES)} = {len(MODELS)}x{len(QUANTILES)}")

In [ ]:
# ============================================================================
# V2: READ STATIC TABLE → PIVOT → SHARED COLUMNS → SCENARIOS
# ============================================================================
# Replaces old cells 3-5b. Everything from one source: forecast_pure_predictions.
# Market-level actuals added from forecast_all_models (forecasts suppressed).
# ============================================================================
from pyspark.sql import functions as F, Window

import pandas as pd, numpy as np

# ============================================================================
# A. READ STATIC TABLE (Total-level, all models, calibrated bands)
# ============================================================================
pure_raw = spark.table(PURE_TABLE).toPandas()
pure_raw['ds'] = pd.to_datetime(pure_raw['ds'])

# Normalize model names
pure_raw['model_name'] = pure_raw['model'].replace(MODEL_RENAME)
print(f"Static table: {len(pure_raw)} rows, {pure_raw['ds'].nunique()} weeks")
print(f"Models (after rename): {sorted(pure_raw['model_name'].unique())}")

# ============================================================================
# B. PIVOT: 6 models × 5 quantiles = 30 model columns
# ============================================================================
KEY_COLS = ['ds', 'product']
pivot_rows = []
for (ds, prod), grp in pure_raw.groupby(KEY_COLS):
    row = {'ds': ds, 'product': prod}
    for _, r in grp.iterrows():
        m = r['model_name']
        if m not in MODELS:
            continue
        for q_label, q_col in QUANTILE_MAP.items():
            row[f'{m}_{q_label}'] = r[q_col]
    # Ensemble metadata for this week
    ens = grp[grp['model_name'] == 'Adaptive_Ensemble']
    if not ens.empty:
        e = ens.iloc[0]
        row['actuals'] = e.get('actual', np.nan)
        row['blended_p50'] = e.get('blended_p50', np.nan)
        row['source'] = e.get('source', None)
        row['bands_method'] = e.get('bands_method', None)
    pivot_rows.append(row)

wide_total = pd.DataFrame(pivot_rows)
wide_total['sales_market'] = 'Total'
wide_total['grain_level'] = 'total'
print(f"Pivoted total rows: {len(wide_total)}")

# ============================================================================
# C. SHARED COLUMNS: scenarios, calendar, forecast_type
# ============================================================================
# Scenarios from Ensemble (real native Adaptive_Ensemble P25/P75, no longer interpolated from p30/p70)
wide_total['most_likely'] = wide_total['blended_p50']  # actual where closed, forecast where future
wide_total['worst_case']  = wide_total['Adaptive_Ensemble_P25']
wide_total['best_case']   = wide_total['Adaptive_Ensemble_P75']

# forecast_type: consistent per date
wide_total['forecast_type'] = np.where(
    wide_total['actuals'].notna(), 'actuals', 'forecast')

# Calendar
wide_total['year']    = wide_total['ds'].dt.year
wide_total['quarter'] = wide_total['ds'].dt.quarter
wide_total['month']   = wide_total['ds'].dt.month
wide_total['ISOWeek'] = wide_total['ds'].dt.isocalendar().week.astype(int)
global_min = wide_total['ds'].min()
wide_total['RelWeekNumber'] = ((wide_total['ds'] - global_min).dt.days // 7) + 1
wide_total['WeekLabel']     = 'W' + wide_total['RelWeekNumber'].astype(str)
wide_total['Product_group'] = wide_total['product']

# model_fill_source from static table's source column
wide_total['model_fill_source'] = wide_total['source']

# ARR_YTD: running sum of actuals within year (NULL past last actual)
wide_total = wide_total.sort_values(['product', 'ds'])
for prod in wide_total['product'].unique():
    mask = wide_total['product'] == prod
    act = wide_total.loc[mask, 'actuals']
    ytd = act.groupby(wide_total.loc[mask, 'year']).cumsum()
    # NULL past last actual
    last_act = wide_total.loc[mask & wide_total['actuals'].notna(), 'ds'].max()
    ytd[wide_total.loc[mask, 'ds'] > last_act] = np.nan
    wide_total.loc[mask, 'ARR_YTD'] = ytd

# mc_actuals and miradore_plan: not in static table, set NULL
wide_total['mc_actuals'] = np.nan
wide_total['miradore_plan'] = np.nan

# ============================================================================
# C1b. OVERRIDE FORWARD FORECAST WITH BOQ-PACED VALUES FROM CBE
# ============================================================================
# Raw p50 is the model output. CBE 'forecast' column is BOQ-paced (conservative).
# Override most_likely for forward weeks. Bands stay from calibrated predictions.
_UCC_CBE_T = f"{CATALOG}.{SCHEMA}.ucc_forecast_cbe"
_ITSG_CBE_T = f"{CATALOG}.{SCHEMA}.itsg_forecast_cbe"
# 2026-10-06: UCC forward weeks come from the frozen BOQ plan
# (ucc_forecast_boq.forecast) to match forecast_app_snapshot. ITSG stays on CBE.
_BOQ_PRODUCTS = {'UCC'}
import datetime as _dt
_BOQ_FY_END = f"{_dt.date.today().year}-12-31"  # current FY only; raw BOQ forecast overstates 2027+ vs p50
_BOQ_T = {'UCC': f"{CATALOG}.{SCHEMA}.ucc_forecast_boq",
          'ITSG': f"{CATALOG}.{SCHEMA}.itsg_forecast_boq"}

_fwd = wide_total['forecast_type'] == 'forecast'
_boq_count = 0
for _prod, _cbe_t in [('UCC', _UCC_CBE_T), ('ITSG', _ITSG_CBE_T)]:
    if _prod in _BOQ_PRODUCTS:
        _src = _BOQ_T[_prod]
        _cbe_f = spark.sql(f"""
            SELECT CAST(ds AS TIMESTAMP) AS ds, forecast AS boq_forecast
            FROM {_src}
            WHERE model = 'Adaptive_Ensemble' AND source = 'forecast'
              AND grain_level = 'total' AND ds <= '{_BOQ_FY_END}'
              AND run_date_utc = (SELECT MAX(run_date_utc) FROM {_src})
        """).toPandas()
        print(f"  {_prod}: forward most_likely from BOQ plan ({_src}), {len(_cbe_f)} weeks")
    else:
        _cbe_f = spark.sql(f"""
            SELECT CAST(ds AS TIMESTAMP) AS ds, forecast AS boq_forecast
            FROM {_cbe_t}
            WHERE model = 'Adaptive_Ensemble' AND source = 'forecast'
              AND run_date_utc = (SELECT MAX(run_date_utc) FROM {_cbe_t})
        """).toPandas()
    _cbe_f['ds'] = pd.to_datetime(_cbe_f['ds'])
    for _, _r in _cbe_f.iterrows():
        _m = (wide_total['product'] == _prod) & (wide_total['ds'] == _r['ds']) & _fwd
        if _m.sum() > 0:
            wide_total.loc[_m, 'most_likely'] = _r['boq_forecast']
            wide_total.loc[_m, 'blended_p50'] = _r['boq_forecast']
            _boq_count += _m.sum()

# Recompute Total forward most_likely
# 2026-09-30 NOTE: this loop iterates `wide_total.loc[_t_fwd, 'ds'].unique()`,
# i.e. dates where a Total forward row ALREADY EXISTS -- it can only update,
# never create, a Total row (same shape as the closed-week gap fixed just
# above). In practice this is now low-risk because forecast_pure_predictions'
# own Total rollup (fixed 2026-09-30 in forecast_unify_publish Cell 42) sums
# whatever product(s) are present per week, so a Total row should already
# exist here whenever UCC or ITSG does. Left as an update-only loop rather
# than adding insert logic on top of insert logic in the same cell -- if this
# ever needs hardening further, mirror the insert-or-update pattern above.
for _prod_lbl in ['Total']:
    _t_fwd = (wide_total['product'] == _prod_lbl) & _fwd
    for _ds in wide_total.loc[_t_fwd, 'ds'].unique():
        _u = wide_total.loc[(wide_total['product'] == 'UCC') & (wide_total['ds'] == _ds) & _fwd, 'most_likely']
        _i = wide_total.loc[(wide_total['product'] == 'ITSG') & (wide_total['ds'] == _ds) & _fwd, 'most_likely']
        if len(_u) > 0 and len(_i) > 0:
            wide_total.loc[(wide_total['product'] == _prod_lbl) & (wide_total['ds'] == _ds) & _fwd, 'most_likely'] = float(_u.iloc[0]) + float(_i.iloc[0])
            wide_total.loc[(wide_total['product'] == _prod_lbl) & (wide_total['ds'] == _ds) & _fwd, 'blended_p50'] = float(_u.iloc[0]) + float(_i.iloc[0])

print(f"BOQ override: {_boq_count} forward rows updated with CBE forecast values")

# ============================================================================
# C2. FILL 2026 ACTUALS GAP (closed weeks not in pure predictions)
# ============================================================================
# Pure predictions only has forward forecast rows. Closed weeks in 2026
# (e.g. Q3 Jul-Sep 14) are missing. Pull from CBE tables.
UCC_CBE_TBL = f"{CATALOG}.{SCHEMA}.ucc_forecast_cbe"
ITSG_CBE_TBL = f"{CATALOG}.{SCHEMA}.itsg_forecast_cbe"

# Overlay CBE actuals onto wide_total — OVERRIDE hindcast rows for closed weeks
# (pure_predictions may have hindcast model data for Jul-Sep, but closed weeks
# need actual revenue in actuals/most_likely/forecast_type)
existing_dates_by_prod = {}
for p in wide_total['product'].unique():
    existing_dates_by_prod[p] = set(wide_total[wide_total['product'] == p]['ds'])

cbe_gap_rows = []
for product, cbe_tbl in [('UCC', UCC_CBE_TBL), ('ITSG', ITSG_CBE_TBL)]:
    cbe_act = spark.sql(f"""
        SELECT CAST(ds AS TIMESTAMP) AS ds, p50
        FROM {cbe_tbl}
        WHERE model = 'Adaptive_Ensemble' AND source = 'actual'
          AND ds >= '2026-01-05'
          AND run_date_utc = (SELECT MAX(run_date_utc) FROM {cbe_tbl})
    """).toPandas()
    cbe_act['ds'] = pd.to_datetime(cbe_act['ds'])

    # For dates already in wide_total (e.g. hindcast rows), OVERRIDE with actuals
    _existing = existing_dates_by_prod.get(product, set())
    _to_override = cbe_act[cbe_act['ds'].isin(_existing)]
    for _, _ov in _to_override.iterrows():
        _mask = (wide_total['product'] == product) & (wide_total['ds'] == _ov['ds'])
        wide_total.loc[_mask, 'actuals'] = _ov['p50']
        wide_total.loc[_mask, 'most_likely'] = _ov['p50']
        wide_total.loc[_mask, 'blended_p50'] = _ov['p50']
        wide_total.loc[_mask, 'worst_case'] = _ov['p50']
        wide_total.loc[_mask, 'best_case'] = _ov['p50']
        wide_total.loc[_mask, 'forecast_type'] = 'actuals'
        wide_total.loc[_mask, 'source'] = 'cbe_actuals'
        wide_total.loc[_mask, 'model_fill_source'] = 'cbe_actuals'
    if len(_to_override) > 0:
        print(f"  {product}: overrode {len(_to_override)} hindcast rows with CBE actuals")

    # For dates NOT in wide_total, add new rows (original logic)
    cbe_act = cbe_act[~cbe_act['ds'].isin(_existing)]
    if len(cbe_act) > 0:
        gap = pd.DataFrame({'ds': cbe_act['ds'], 'product': product})
        gap['sales_market'] = 'Total'
        gap['grain_level'] = 'total'
        gap['actuals'] = cbe_act['p50'].values
        gap['most_likely'] = cbe_act['p50'].values
        gap['blended_p50'] = cbe_act['p50'].values
        gap['forecast_type'] = 'actuals'
        gap['source'] = 'cbe_actuals'
        gap['model_fill_source'] = 'cbe_actuals'
        gap['year'] = gap['ds'].dt.year
        gap['quarter'] = gap['ds'].dt.quarter
        gap['month'] = gap['ds'].dt.month
        gap['ISOWeek'] = gap['ds'].dt.isocalendar().week.astype(int)
        gap['RelWeekNumber'] = ((gap['ds'] - global_min).dt.days // 7) + 1
        gap['WeekLabel'] = 'W' + gap['RelWeekNumber'].astype(str)
        gap['Product_group'] = product
        gap['mc_actuals'] = np.nan
        gap['miradore_plan'] = np.nan
        gap['worst_case'] = cbe_act['p50'].values
        gap['best_case'] = cbe_act['p50'].values
        # ── Fill model columns from VINTAGE predictions (backtest) ────
        _VINT_TBL = f"{CATALOG}.{SCHEMA}.{'ucc' if product == 'UCC' else 'itsg'}_forecast_vintages"
        _gap_dates_str = ','.join([f"'{d.strftime('%Y-%m-%d')}'" for d in cbe_act['ds']])
        try:
            # Vintages only have p10, p50, p90 natively (this append-only
            # history table was never part of the P05/P25/P50/P75/P95
            # conversion) — approximate p05/p25/p75/p95 from them below.
            _vint_df = spark.sql(f"""
                WITH ranked AS (
                    SELECT ds, model, p10, p50, p90,
                           ROW_NUMBER() OVER (PARTITION BY ds, model ORDER BY run_date_utc ASC) AS rn
                    FROM {_VINT_TBL}
                    WHERE grain_level = 'total' AND ds IN ({_gap_dates_str})
                )
                SELECT CAST(ds AS TIMESTAMP) AS ds, model, p10, p50, p90
                FROM ranked WHERE rn = 1
            """).toPandas()
            _vint_df['ds'] = pd.to_datetime(_vint_df['ds'])
            _VINT_MODEL_RENAME = MODEL_RENAME
            _vint_df['model_name'] = _vint_df['model'].replace(_VINT_MODEL_RENAME)
            # Synthesize p10/p90 for models that only store p50 in vintages,
            # using Adaptive_Ensemble's relative spread for that week.
            for _ds_val in _vint_df['ds'].unique():
                _ens_row = _vint_df[(_vint_df['ds'] == _ds_val) & (_vint_df['model_name'] == 'Adaptive_Ensemble')]
                if _ens_row.empty or pd.isna(_ens_row.iloc[0]['p10']):
                    continue
                _ens_p50 = _ens_row.iloc[0]['p50']
                if _ens_p50 == 0 or pd.isna(_ens_p50):
                    continue
                _r_low  = _ens_row.iloc[0]['p10'] / _ens_p50   # e.g. 0.66
                _r_high = _ens_row.iloc[0]['p90'] / _ens_p50   # e.g. 1.37
                _null_mask = (_vint_df['ds'] == _ds_val) & _vint_df['p10'].isna()
                _vint_df.loc[_null_mask, 'p10'] = _vint_df.loc[_null_mask, 'p50'] * _r_low
                _vint_df.loc[_null_mask, 'p90'] = _vint_df.loc[_null_mask, 'p50'] * _r_high
            # Approximate p05/p25/p75/p95 from native p10/p50/p90 (this
            # append-only vintages table only ever stores p10/p50/p90).
            # p25/p75: linear interpolation between p50 and p10/p90.
            # p05/p95: widen the p10/p90 half-width by the Z-score ratio
            # (Z_05/Z_10 = 1.6449/1.2816 ≈ 1.284), consistent with the
            # normal-approximation approach used elsewhere in this pipeline
            # (e.g. forecast_unify_publish's ITSG backfill, now removed, used
            # the same Z-ratio method).
            _Z_RATIO_05_10 = 1.6449 / 1.2816
            _vint_df['p25'] = (_vint_df['p10'] + _vint_df['p50']) / 2
            _vint_df['p75'] = (_vint_df['p50'] + _vint_df['p90']) / 2
            _vint_df['p05'] = _vint_df['p50'] - (_vint_df['p50'] - _vint_df['p10']) * _Z_RATIO_05_10
            _vint_df['p95'] = _vint_df['p50'] + (_vint_df['p90'] - _vint_df['p50']) * _Z_RATIO_05_10
            # Pivot vintage predictions into wide model columns
            for _, _vr in _vint_df.iterrows():
                _m = _vr['model_name']
                if _m not in MODELS:
                    continue
                _ds_match = gap['ds'] == _vr['ds']
                for _q_label, _q_col in QUANTILE_MAP.items():
                    gap.loc[_ds_match, f'{_m}_{_q_label}'] = _vr[_q_col]
            print(f"  {product}: {len(cbe_act)} actuals + vintage backtest model predictions")
        except Exception as _ve:
            print(f"  {product}: {len(cbe_act)} actuals (vintage lookup failed: {_ve})")
        # ARR_YTD will be recomputed after concat
        cbe_gap_rows.append(gap)
        print(f"  {product}: gap rows ready")

if cbe_gap_rows:
    cbe_gap_df = pd.concat(cbe_gap_rows, ignore_index=True)
    wide_total = pd.concat([wide_total, cbe_gap_df], ignore_index=True)
    wide_total = wide_total.sort_values(['product', 'ds']).reset_index(drop=True)
    print(f"  Wide total after gap fill: {len(wide_total)} rows")
else:
    print("  No new actuals gap rows needed (existing rows overridden in place)")

# ── Propagate actuals to Total product (UCC + ITSG) for overridden weeks ──
# 2026-09-30 FIX: this used to only UPDATE an existing Total row (`if
# _t_mask.sum() > 0`), with no else branch to INSERT one -- the exact same
# bug shape as the Total-row gap already found and fixed in
# forecast_unify_publish's forecast_app_snapshot cell. If a date has real
# UCC+ITSG actuals but `pure_raw`/`wide_total` never got a Total row for it
# (e.g. an upstream table gap), that date's Total silently had no row at all
# in arr_forecast_powerbi_wide/arr_forecast_long. Now inserts a new Total row
# when none exists, instead of only ever overriding one already there.
_ucc_act = wide_total[(wide_total['product'] == 'UCC') & (wide_total['forecast_type'] == 'actuals')]
_itsg_act = wide_total[(wide_total['product'] == 'ITSG') & (wide_total['forecast_type'] == 'actuals')]
_act_dates = set(_ucc_act['ds']) & set(_itsg_act['ds'])
_total_overrides = 0
_total_inserts = 0
_total_new_rows = []
_total_template = wide_total[wide_total['product'] == 'Total'].iloc[0].to_dict() if (wide_total['product'] == 'Total').any() else None
for _d in _act_dates:
    _t_mask = (wide_total['product'] == 'Total') & (wide_total['ds'] == _d)
    _u_val = float(_ucc_act.loc[_ucc_act['ds'] == _d, 'actuals'].iloc[0])
    _i_val = float(_itsg_act.loc[_itsg_act['ds'] == _d, 'actuals'].iloc[0])
    _total_val = _u_val + _i_val
    if _t_mask.sum() > 0:
        wide_total.loc[_t_mask, 'actuals'] = _total_val
        wide_total.loc[_t_mask, 'most_likely'] = _total_val
        wide_total.loc[_t_mask, 'blended_p50'] = _total_val
        wide_total.loc[_t_mask, 'worst_case'] = _total_val
        wide_total.loc[_t_mask, 'best_case'] = _total_val
        wide_total.loc[_t_mask, 'forecast_type'] = 'actuals'
        wide_total.loc[_t_mask, 'source'] = 'cbe_actuals'
        wide_total.loc[_t_mask, 'model_fill_source'] = 'cbe_actuals'
        _total_overrides += 1
    elif _total_template is not None:
        # No Total row exists for this date at all -- build one from a
        # template Total row's column shape (so every existing column, incl.
        # per-model quantile columns, is present), null out anything we don't
        # actually know for this synthesized row, then set the real fields.
        _new_row = dict(_total_template)
        for _k in _new_row:
            if _k not in ('product', 'sales_market', 'grain_level'):
                _new_row[_k] = np.nan
        _new_row['ds'] = _d
        _new_row['product'] = 'Total'
        _new_row['actuals'] = _total_val
        _new_row['most_likely'] = _total_val
        _new_row['blended_p50'] = _total_val
        _new_row['worst_case'] = _total_val
        _new_row['best_case'] = _total_val
        _new_row['forecast_type'] = 'actuals'
        _new_row['source'] = 'cbe_actuals'
        _new_row['model_fill_source'] = 'cbe_actuals'
        _new_row['year'] = _d.year
        _new_row['quarter'] = (_d.month - 1) // 3 + 1
        _new_row['month'] = _d.month
        _new_row['ISOWeek'] = _d.isocalendar()[1]
        _new_row['RelWeekNumber'] = ((_d - global_min).days // 7) + 1
        _new_row['WeekLabel'] = 'W' + str(_new_row['RelWeekNumber'])
        _new_row['Product_group'] = 'Total'
        _total_new_rows.append(_new_row)
        _total_inserts += 1
if _total_new_rows:
    wide_total = pd.concat([wide_total, pd.DataFrame(_total_new_rows)], ignore_index=True)
    wide_total = wide_total.sort_values(['product', 'ds']).reset_index(drop=True)
if _total_overrides > 0 or _total_inserts > 0:
    print(f"  Total: overrode {_total_overrides} existing rows + inserted {_total_inserts} new rows with UCC+ITSG actuals sum")

# ── Always recompute ARR_YTD (covers both overridden and newly-added actuals) ──
wide_total = wide_total.sort_values(['product', 'ds']).reset_index(drop=True)
for prod in wide_total['product'].unique():
    mask = wide_total['product'] == prod
    act = wide_total.loc[mask, 'actuals']
    ytd = act.groupby(wide_total.loc[mask, 'year']).cumsum()
    last_act = wide_total.loc[mask & wide_total['actuals'].notna(), 'ds'].max()
    if pd.notna(last_act):
        ytd[wide_total.loc[mask, 'ds'] > last_act] = np.nan
    wide_total.loc[mask, 'ARR_YTD'] = ytd
print(f"  ARR_YTD recomputed for all products")

# ============================================================================
# C3. EXPLICIT STATIC / DYNAMIC FORECAST PAIR (Total grain)
# ============================================================================
# 2026-09-30: added for the Power BI "static forecast vs dynamic forecast"
# comparison view (frozen plan vs current-best-estimate). Both values already
# existed under less obvious names -- these are self-explanatory aliases so a
# report author doesn't have to know that convention:
#   Static_P50  = Adaptive_Ensemble_P50 -- the model's original prediction.
#     Never overwritten once published, including for closed weeks (this now
#     holds end-to-end thanks to the 2026-09-30 fix in forecast_unify_publish's
#     PURE FORECAST PREDICTIONS cell, which stopped a closed week's actual
#     value from silently replacing the model's p05-p95 for UCC).
#   Dynamic_P50 = blended_p50 / most_likely -- actual where the week has
#     closed, Static_P50 where it hasn't. Computed AFTER the BOQ-paced
#     override (C1b) and the actuals gap-fill (C2) above, so it reflects the
#     final current-best-estimate value, not an intermediate one.
wide_total['Static_P50']  = wide_total['Adaptive_Ensemble_P50']
wide_total['Dynamic_P50'] = wide_total['blended_p50']
_static_nulls = wide_total['Static_P50'].isna().sum()
_dynamic_nulls = wide_total['Dynamic_P50'].isna().sum()
print(f"  Static_P50 / Dynamic_P50 added (Total grain): "
      f"{_static_nulls} static nulls, {_dynamic_nulls} dynamic nulls out of {len(wide_total)} rows")


# ============================================================================
# D. MARKET-LEVEL ACTUALS (from forecast_all_models, forecasts suppressed)
# ============================================================================
mkt = spark.sql(f"""
    SELECT CAST(ds AS DATE) AS ds, product,
           -- 2026-09-30 FIX: COALESCE at the source -- a genuine SQL NULL
           -- sales_market (not the string 'UNKNOWN', which the .replace()
           -- below already handled) was passing straight through ungrouped,
           -- producing market-grain rows with no market label at all (394
           -- of 1,928 market rows). Collapse true NULLs into the same
           -- 'UNKNOWN' bucket the rest of this pipeline already recognizes.
           COALESCE(sales_market, 'UNKNOWN') AS sales_market,
           MAX(actuals) AS actuals,
           MAX(magna_carta_actuals) AS mc_actuals,
           MAX(miradore_plan) AS miradore_plan
    FROM {MARKET_SOURCE}
    WHERE is_current = TRUE AND actuals IS NOT NULL
      AND sales_market != 'Total'
    GROUP BY ds, product, COALESCE(sales_market, 'UNKNOWN')
""").toPandas()
mkt['ds'] = pd.to_datetime(mkt['ds'])
mkt['grain_level'] = 'market'
mkt['forecast_type'] = 'actuals'
mkt['sales_market'] = mkt['sales_market'].replace({'UNKNOWN': 'Unknown'})
for col in ['year', 'quarter', 'month']:
    mkt[col] = getattr(mkt['ds'].dt, col if col != 'quarter' else col)
mkt['year'] = mkt['ds'].dt.year
mkt['quarter'] = mkt['ds'].dt.quarter
mkt['month'] = mkt['ds'].dt.month
mkt['ISOWeek'] = mkt['ds'].dt.isocalendar().week.astype(int)
mkt['RelWeekNumber'] = ((mkt['ds'] - global_min).dt.days // 7) + 1
mkt['WeekLabel'] = 'W' + mkt['RelWeekNumber'].astype(str)
mkt['Product_group'] = mkt['product']
mkt['model_fill_source'] = 'historical'
print(f"Market-level actuals: {len(mkt)} rows")

# ============================================================================
# E. UNION TOTAL + MARKET → STAGING VIEW
# ============================================================================
wide = pd.concat([wide_total, mkt], ignore_index=True)
wide = wide.sort_values(['product', 'sales_market', 'ds']).reset_index(drop=True)

# Metadata
wide['run_date'] = pd.Timestamp.today().normalize()
wide['published_at_utc'] = pd.Timestamp.utcnow()

# Null model columns at market grain (forecasts unreliable)
model_cols = [c for c in wide.columns if any(c.startswith(m + '_') for m in MODELS)]
scenario_cols = ['most_likely', 'worst_case', 'best_case', 'ARR_YTD']
mkt_mask = wide['grain_level'] == 'market'
for c in model_cols + scenario_cols:
    if c in wide.columns:
        wide.loc[mkt_mask, c] = np.nan

print(f"\nWide rows: {len(wide)}, columns: {len(wide.columns)}")
print(f"  Total-level: {(wide['grain_level']=='total').sum()}, Market-level: {mkt_mask.sum()}")
print(f"  Actuals: {(wide['forecast_type']=='actuals').sum()}, Forecast: {(wide['forecast_type']=='forecast').sum()}")

# Register as staging view for SQL write
sdf = spark.createDataFrame(wide)
sdf.createOrReplaceTempView('_powerbi_wide_staging')
print(f"\u2713 Registered _powerbi_wide_staging")

# Bypass old cells 4-5b (all logic consolidated here)
pivoted = sdf  # compatibility if old cells reference this
print('Old cells 4-5b are now no-ops — all processing done here.')

In [ ]:
# ============================================================================
# [V2 NO-OP] Old pivot logic consolidated into Cell 4 above.
# ============================================================================
pass  # All pivot logic now in Cell 4
# OLD CODE BELOW (kept for reference, not executed)
if False:  # pragma: no cover
    _OLD = """
# ============================================================================
# PIVOT: 7 models x 5 quantiles = 35 model columns
# ============================================================================
# Each quantile gets its own pivot pass, then all are joined on the natural key.

KEY_COLS = ["ds", "product", "sales_market"]
CALENDAR_COLS = ["year", "ISOWeek", "quarter", "month", "RelWeekNumber", "WeekLabel", "Product_group"]
GROUP_COLS = KEY_COLS + CALENDAR_COLS

quantile_map = {
    "P05": "forecast_p05",
    "P25": "forecast_p25",
    "P50": "forecast_p50",
    "P75": "forecast_p75",
    "P95": "forecast_p95",
}

pivoted = None
for q_label, q_col in quantile_map.items():
    piv = (
        src
        .groupBy(*GROUP_COLS)
        .pivot("model_name", values=MODELS)
        .agg(F.first(q_col))
    )
    # Rename pivoted columns to {Model}_{Quantile}
    for m in MODELS:
        if m in piv.columns:
            piv = piv.withColumnRenamed(m, f"{m}_{q_label}")

    if pivoted is None:
        pivoted = piv
    else:
        pivoted = pivoted.join(piv, on=GROUP_COLS, how="outer")

model_cols = [c for c in pivoted.columns if c not in GROUP_COLS]
print(f"Pivoted rows: {pivoted.count()}")
print(f"Model columns ({len(model_cols)}): {sorted(model_cols)[:10]} ...")
assert len(model_cols) == len(MODELS) * len(QUANTILES), (
    f"Expected {len(MODELS)*len(QUANTILES)} model cols, got {len(model_cols)}"
)
"""

In [ ]:
# ============================================================================
# [V2 NO-OP] Old shared-columns + backfill logic consolidated into Cell 4.
# ============================================================================
pass
print('V2: Shared columns, scenarios, backfill, YTD all handled in Cell 4.')
if False:  # pragma: no cover
    _OLD = """
# ============================================================================
# SHARED (non-pivoted) COLUMNS
# ============================================================================

# --- Actuals + driver features (model-independent, first non-null) ---
shared = (
    src
    .groupBy(*KEY_COLS)
    .agg(
        F.first("actuals", ignorenulls=True).alias("actuals_raw"),
        F.first("magna_carta_actuals", ignorenulls=True).alias("mc_actuals"),
        F.first("miradore_plan", ignorenulls=True).alias("miradore_plan"),
        # FIX Issue 4: Consolidate forecast_type — priority: actuals > rolling > forecast
        F.first("forecast_type", ignorenulls=True).alias("forecast_type_raw"),
    )
)

# --- Join shared onto pivoted ---
wide = pivoted.join(shared, on=KEY_COLS, how="left")

# ============================================================================
# FIX Issue 3+4: ACTUALS BOUNDARY + CONSISTENT forecast_type
# ============================================================================
# The source has staggered actual dates per slice (ITSG×EMEA extends to
# 2026-09-21 while others stop at 2026-08-24). We compute a global cutoff:
# the MEDIAN last-actual date across all market-level slices, so one slice's
# late ETL doesn't leak future actuals into the wide table.

# Compute last non-zero actual date per product × market
last_actual_per_slice = (
    src.filter(
        (F.col("actuals").isNotNull()) & (F.col("actuals") != 0) &
        (F.col("sales_market") != "Total") & (F.col("product") != "Total")
    )
    .groupBy("product", "sales_market")
    .agg(F.max("ds").alias("last_actual_ds"))
)

# Use the most common (mode) cutoff as the global boundary
from pyspark.sql.types import DateType
global_cutoff = (
    last_actual_per_slice
    .groupBy("last_actual_ds")
    .count()
    .orderBy(F.col("count").desc())
    .first()["last_actual_ds"]
)
print(f"Global actuals cutoff: {global_cutoff}")

# Null out actuals that leak past the cutoff
wide = wide.withColumn("actuals",
    F.when(F.col("ds") <= F.lit(global_cutoff), F.col("actuals_raw"))
     .otherwise(F.lit(None))
).drop("actuals_raw")

# Assign consistent forecast_type per row
wide = wide.withColumn("forecast_type",
    F.when(F.col("ds") <= F.lit(global_cutoff), F.lit("actuals"))
     .otherwise(F.lit("forecast"))
).drop("forecast_type_raw")

print(f"Actuals rows: {wide.filter(F.col('forecast_type') == 'actuals').count()}, "
      f"Forecast rows: {wide.filter(F.col('forecast_type') == 'forecast').count()}")

# ============================================================================
# BACKFILL HISTORICAL GAPS: use hindcasts first, then Prophet proxy fallback
# ============================================================================
# Closed-quarter rows (Q1/Q2 2026) predate the native V5/V9 launches, so
# forecast_all_models only carries Prophet values there. We now backfill those
# product-total historical rows from the retroactive hindcast tables first;
# only rows with no native model AND no hindcast still fall back to Prophet.
NON_PROPHET_MODELS = [m for m in MODELS if m != "Prophet"]
HINDCAST_MODEL_MAP = {
    "Adaptive_Ensemble": "Adaptive_Ensemble",
    "DHR_ARIMA": "DHR_ARIMA",
    "ETS": "ETS",
    "Global_LGB_Q50_ITSG": "Global_LGB_Q50_ITSG",
    "Global_LGB_Q50_UCC": "Global_LGB_Q50_UCC",
    "MSTL_v2": "MSTL_v2",
    "Prophet": "Prophet_trend",
}

v5_any_filled = F.lit(False)
for m in NON_PROPHET_MODELS:
    v5_any_filled = v5_any_filled | F.col(f"{m}_P50").isNotNull()


def build_hindcast_backfill(table_name: str, product_name: str):
    hc = (
        spark.table(table_name)
        .select(F.to_date("ds").alias("ds"), "model", F.col("forecast").cast("double").alias("forecast"))
        # Reruns append duplicate rows; average keeps the notebook idempotent
        # without assuming a sortable pipeline_version naming scheme.
        .groupBy("ds", "model")
        .agg(F.avg("forecast").alias("forecast"))
        .groupBy("ds")
        .pivot("model", values=list(HINDCAST_MODEL_MAP.values()))
        .agg(F.first("forecast"))
    )
    for export_model, hindcast_model in HINDCAST_MODEL_MAP.items():
        out_col = f"hc_{export_model}_P50"
        if hindcast_model in hc.columns:
            hc = hc.withColumnRenamed(hindcast_model, out_col)
        else:
            hc = hc.withColumn(out_col, F.lit(None).cast("double"))
    return hc.select("ds", *[f"hc_{m}_P50" for m in MODELS]).withColumn("product", F.lit(product_name))


hindcast_backfill = build_hindcast_backfill(UCC_HINDCAST_TABLE, "UCC").unionByName(
    build_hindcast_backfill(ITSG_HINDCAST_TABLE, "ITSG")
)
wide = wide.join(hindcast_backfill, on=["ds", "product"], how="left")

hindcast_any_filled = F.lit(False)
for m in MODELS:
    hindcast_any_filled = hindcast_any_filled | F.col(f"hc_{m}_P50").isNotNull()

is_hindcast_backfill_row = (
    (F.col("sales_market") == "Total") &
    (F.col("product") != "Total") &
    (F.col("forecast_type") == "actuals") &
    hindcast_any_filled
)

wide = wide.withColumn(
    "model_fill_source",
    F.when(v5_any_filled, F.lit("native"))
     .when(is_hindcast_backfill_row, F.lit("hindcast_point"))
     .otherwise(F.lit("prophet_proxy"))
)

for m in MODELS:
    hc_col = f"hc_{m}_P50"
    for q in QUANTILES:
        col_name = f"{m}_{q}"
        prophet_col = f"Prophet_{q}"
        wide = wide.withColumn(
            col_name,
            F.when(is_hindcast_backfill_row & F.col(hc_col).isNotNull(), F.col(hc_col))
             .otherwise(F.coalesce(F.col(col_name), F.col(prophet_col)))
        )

wide = wide.drop(*[f"hc_{m}_P50" for m in MODELS])

hindcast_count = wide.filter(F.col("model_fill_source") == "hindcast_point").count()
filled_count = wide.filter(F.col("model_fill_source") == "prophet_proxy").count()
native_count = wide.filter(F.col("model_fill_source") == "native").count()
print(
    f"Hindcast backfilled rows: {hindcast_count}, "
    f"Prophet-proxy rows: {filled_count}, Native rows: {native_count}"
)

# ============================================================================
# FIX Issue 2 (negative forecasts): Floor all P05 columns at 0
# ============================================================================
for m in MODELS:
    p05_col = f"{m}_P05"
    wide = wide.withColumn(p05_col, F.greatest(F.col(p05_col), F.lit(0.0)))

neg_check = sum(wide.filter(F.col(f"{m}_P05") < 0).count() for m in MODELS)
print(f"Remaining negative P05 values: {neg_check}")

# --- Scenario columns from Adaptive_Ensemble (now filled + floored) ---
wide = (
    wide
    .withColumn("most_likely", F.col("Adaptive_Ensemble_P50"))
    .withColumn("best_case",   F.col("Adaptive_Ensemble_P95"))
    .withColumn("worst_case",  F.col("Adaptive_Ensemble_P05"))
)

# ============================================================================
# FIX Issue 5: ARR_YTD — null past last actual, don't flatline
# ============================================================================
ytd_window = (
    Window
    .partitionBy("product", "sales_market", "year")
    .orderBy("ds")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)
# Only sum actuals (already nulled past cutoff), then null YTD in forecast zone
wide = wide.withColumn("ARR_YTD",
    F.when(F.col("forecast_type") == "actuals",
           F.sum("actuals").over(ytd_window))
     .otherwise(F.lit(None))
)

# --- Miradore-adjusted scenarios ---
wide = (
    wide
    .withColumn("Best_Case_Miradore_Adj",
                F.col("best_case") + F.coalesce(F.col("miradore_plan"), F.lit(0)))
    .withColumn("Most_Likely_Miradore_Adj",
                F.col("most_likely") + F.coalesce(F.col("miradore_plan"), F.lit(0)))
    .withColumn("Worst_Case_Miradore_Adj",
                F.col("worst_case") + F.coalesce(F.col("miradore_plan"), F.lit(0)))
)
"""

# V2: rebind to Spark DF created in Cell 3
wide = sdf

# ============================================================================
# FIX Issue 7: mc_actuals flag (monthly broadcast, not weekly)
# ============================================================================
wide = wide.withColumn("mc_actuals_is_monthly", F.lit(True))

# ============================================================================
# FIX Issue 1: CI band label (updated: conformal recalibration in Cell 5b
# achieves ~90% coverage, so this label is now informational context).
# ============================================================================
wide = wide.withColumn("ci_band_label",
    F.lit("P05-P95 (conformal-recalibrated per product x market, ~90% empirical coverage)"))

# ============================================================================
# ACCURACY METADATA: Surface real holdout WAPE from monitoring baseline
# ============================================================================
# IMPORTANT: The MAPE figures often cited (market ~1,915%, total ~49%) are
# measured by comparing prophet-proxy forecast columns against actuals — but
# proxy rows were NEVER genuine forecasts for those weeks.  The real forecast
# accuracy is the Adaptive Ensemble holdout WAPE from the UCC Forecast
# Foundation's walk-forward evaluation.
#
# is_native_model = True ONLY for forecast-period, native-V5 rows — the
# only rows where comparing forecast vs eventual actuals is valid.
# (V9 called this "accuracy_eligible" — renamed for clarity: it marks rows
# backed by a real model run, not rows that have been accuracy-tested.)

wide = wide.withColumn("is_native_model",
    (F.col("forecast_type") == "forecast") & (F.col("model_fill_source") == "native")
)

# Read the latest ensemble holdout WAPE from the monitoring baseline
_ucc_baseline = spark.table(f"{CATALOG}.{SCHEMA}.ucc_forecast_monitoring_baseline")
_latest_bl_date = _ucc_baseline.agg(F.max("run_date_utc")).collect()[0][0]
_ensemble_row = (
    _ucc_baseline.filter(
        (F.col("run_date_utc") == _latest_bl_date) &
        (F.col("model") == "Adaptive_Ensemble")
    )
    .select("holdout_wape_baseline")
    .collect()
)
_ensemble_wape = round(_ensemble_row[0]["holdout_wape_baseline"], 1) if _ensemble_row else None
_holdout_start = "2026-01-05"  # Known UCC holdout start (monitoring baseline doesn't track this column)
wide = wide.withColumn("ensemble_holdout_wape", F.lit(_ensemble_wape))
wide = wide.withColumn("holdout_eval_start", F.lit(_holdout_start).cast("date"))
print(f"Ensemble holdout WAPE (from monitoring baseline, {_latest_bl_date}): {_ensemble_wape}%")
print(f"Holdout evaluation period starts: {_holdout_start}")
print(f"Native-model rows: {wide.filter(F.col('is_native_model')).count()} "
      f"(of {wide.filter(F.col('forecast_type') == 'forecast').count()} forecast rows)")

print(f"Wide rows: {wide.count()}, columns: {len(wide.columns)}")

In [ ]:
# ============================================================================
# [V2 NO-OP] Old conformal recalibration + top-down reconciliation + LGB merge.
# All calibration now handled upstream in forecast_pure_predictions.
# ============================================================================
pass
print('V2: Conformal calibration is upstream. LGB merge done in Cell 4. Reconciliation dropped (market forecasts suppressed).')
if False:  # pragma: no cover
    import numpy as np
    from pyspark.sql.types import DoubleType, StringType, StructType, StructField
    _OLD = """
# A. CONFORMAL RECALIBRATION: Replace model-native P05/P95 with empirical bands
# ============================================================================
# Prophet P05-P95 only covers ~30% of actuals — basically useless as a CI.
#
# We use ABSOLUTE residuals (actual - P50), not percentage-based, because
# ~185 rows have P50=0.0 (sparse slices like Unknown). Percentage-based
# conformal collapses to [0, 0] for those rows and dramatically under-covers.
#
# Absolute conformal: P05 = P50 + abs_q05, P95 = P50 + abs_q95
# Per product x market calibration ensures ~90% coverage per slice.

prophet_abs = (
    src.filter(
        (F.col("model_name") == "Prophet") &
        (F.col("actuals").isNotNull()) &
        (F.col("forecast_p50").isNotNull())
        # Total-grain rows included so they get their own conformal bands.
        # Without this, Total P05/P95 collapse to P50 (zero-width CI).
    )
    .withColumn("abs_residual", F.col("actuals") - F.col("forecast_p50"))
)

# Per-product x per-market ABSOLUTE conformal factors
conformal_rows = []
for row in prophet_abs.select("product", "sales_market").distinct().collect():
    prod, mkt = row["product"], row["sales_market"]
    resids = prophet_abs.filter(
        (F.col("product") == prod) & (F.col("sales_market") == mkt)
    ).select("abs_residual").toPandas()["abs_residual"].dropna().values
    if len(resids) >= 10:
        q05, q25, q75, q95 = [float(np.percentile(resids, p)) for p in [5, 25, 75, 95]]
        cov = float(np.mean((resids >= q05) & (resids <= q95)))
        conformal_rows.append((prod, mkt, q05, q25, q75, q95, len(resids), cov))
        print(f"  {prod}|{mkt}: n={len(resids):3d}, q05={q05:+12,.0f}, q95={q95:+12,.0f}, cov={cov:.1%}")

# Broadcast conformal factors as a Spark DataFrame for join
cf_schema = StructType([
    StructField("cf_product", StringType()),
    StructField("cf_market", StringType()),
    StructField("cf_q05", DoubleType()),
    StructField("cf_q25", DoubleType()),
    StructField("cf_q75", DoubleType()),
    StructField("cf_q95", DoubleType()),
    StructField("cf_n", DoubleType()),
    StructField("cf_cov", DoubleType()),
])
cf_df = spark.createDataFrame(
    [(r[0], r[1], r[2], r[3], r[4], r[5], float(r[6]), r[7]) for r in conformal_rows],
    schema=cf_schema
)

# Join conformal factors onto wide table
wide = wide.join(
    cf_df,
    (wide["product"] == cf_df["cf_product"]) & (wide["sales_market"] == cf_df["cf_market"]),
    how="left"
).drop("cf_product", "cf_market", "cf_n", "cf_cov")

# Apply ABSOLUTE conformal bands: P05 = P50 + abs_q05, P95 = P50 + abs_q95
for m in MODELS:
    p50_col = f"{m}_P50"
    wide = wide.withColumn(f"{m}_P05",
        F.greatest(F.col(p50_col) + F.coalesce(F.col("cf_q05"), F.lit(0.0)), F.lit(0.0))
    )
    wide = wide.withColumn(f"{m}_P25",
        F.col(p50_col) + F.coalesce(F.col("cf_q25"), F.lit(0.0))
    )
    wide = wide.withColumn(f"{m}_P75",
        F.col(p50_col) + F.coalesce(F.col("cf_q75"), F.lit(0.0))
    )
    wide = wide.withColumn(f"{m}_P95",
        F.col(p50_col) + F.coalesce(F.col("cf_q95"), F.lit(0.0))
    )

wide = wide.drop("cf_q05", "cf_q25", "cf_q75", "cf_q95")

# Update scenario columns with recalibrated bands
wide = (
    wide
    .withColumn("worst_case",  F.col("Adaptive_Ensemble_P05"))
    .withColumn("best_case",   F.col("Adaptive_Ensemble_P95"))
    .withColumn("Best_Case_Miradore_Adj",
                F.col("best_case") + F.coalesce(F.col("miradore_plan"), F.lit(0)))
    .withColumn("Worst_Case_Miradore_Adj",
                F.col("worst_case") + F.coalesce(F.col("miradore_plan"), F.lit(0)))
)

# Replace ci_band_label
wide = wide.withColumn("ci_band_label",
    F.lit("Conformal P05-P95 (calibrated from Prophet residuals, ~90% empirical coverage)"))

print("Conformal recalibration applied to all 7 models.")

# ============================================================================
# B. TOP-DOWN RECONCILIATION: Product-Total forecast × market share
# ============================================================================
# Market-level forecasts are very noisy (MAPE ~1700%).
# The product×Total forecast is much less noisy (MAPE ~28%).
# We reconcile: reconciled_market = Total_forecast × historical_market_share.

# Get product×Total rows for Adaptive_Ensemble
total_forecasts = (
    wide.filter((F.col("sales_market") == "Total") & (F.col("product") != "Total"))
    .select(
        "ds", "product",
        F.col("Adaptive_Ensemble_P50").alias("total_p50"),
        F.col("Adaptive_Ensemble_P05").alias("total_p05"),
        F.col("Adaptive_Ensemble_P95").alias("total_p95"),
    )
)

# Market share from historical actuals
mkt_shares = (
    src.filter(
        (F.col("model_name") == "Prophet") &
        (F.col("actuals").isNotNull()) & (F.col("actuals") > 0) &
        (F.col("sales_market") != "Total") & (F.col("product") != "Total")
    )
    .groupBy("product", "sales_market")
    .agg(F.sum("actuals").alias("slice_actuals"))
)
prod_totals = mkt_shares.groupBy("product").agg(F.sum("slice_actuals").alias("prod_total"))
shares = (
    mkt_shares.join(prod_totals, "product")
    .withColumn("market_share", F.col("slice_actuals") / F.col("prod_total"))
    .select("product", "sales_market", "market_share")
)

# Join Total forecasts × market shares onto wide table
wide = wide.join(
    total_forecasts.join(shares, "product"),
    on=["ds", "product", "sales_market"],
    how="left"
)

# Reconciled columns: Total forecast × market share
wide = (
    wide
    .withColumn("reconciled_p50", F.col("total_p50") * F.col("market_share"))
    .withColumn("reconciled_p05", F.greatest(F.col("total_p05") * F.col("market_share"), F.lit(0.0)))
    .withColumn("reconciled_p95", F.col("total_p95") * F.col("market_share"))
    .drop("total_p50", "total_p05", "total_p95", "market_share")
)

reconciled_filled = wide.filter(F.col("reconciled_p50").isNotNull()).count()
print(f"Reconciled columns populated: {reconciled_filled} market-level rows "
      f"(NULL for Total/Total-product rows by design)")

# ============================================================================
# C. MERGE LGB COLUMNS: Combine ITSG/UCC into single Global_LGB_Q50
# ============================================================================
for q in QUANTILES:
    itsg_col = f"Global_LGB_Q50_ITSG_{q}"
    ucc_col = f"Global_LGB_Q50_UCC_{q}"
    merged_col = f"Global_LGB_Q50_{q}"
    wide = wide.withColumn(merged_col,
        F.when(F.col("product") == "UCC", F.col(ucc_col))
         .when(F.col("product") == "ITSG", F.col(itsg_col))
         .otherwise(F.coalesce(F.col(ucc_col), F.col(itsg_col)))
    )

# Drop the redundant per-product LGB columns
for q in QUANTILES:
    wide = wide.drop(f"Global_LGB_Q50_ITSG_{q}", f"Global_LGB_Q50_UCC_{q}")

print(f"Merged Global_LGB_Q50 columns (5 quantiles). Dropped 10 redundant ITSG/UCC columns.")

# ============================================================================
# D. GRAND TOTAL SYNTHESIS: Sum product-level Total bands
# ============================================================================
# Grand Total (product=Total, sales_market=Total) has no Prophet residuals,
# so conformal factors are NULL → bands collapse to zero-width.
# Fix: synthesize Grand Total bands as SUM of product-level Total bands.
# This is conservative (sum of P05s ≤ joint P05) but far better than 0-width.

FINAL_MODELS = ["Adaptive_Ensemble", "DHR_ARIMA", "ETS", "Global_LGB_Q50", "MSTL_v2", "Prophet"]

# Aggregate product-level Total bands into Grand Total
agg_exprs = []
for m in FINAL_MODELS:
    for q in QUANTILES:
        agg_exprs.append(F.sum(f"{m}_{q}").alias(f"gt_{m}_{q}"))

product_total_bands = (
    wide.filter((F.col("sales_market") == "Total") & (F.col("product") != "Total"))
    .groupBy("ds")
    .agg(*agg_exprs)
)

wide = wide.join(product_total_bands, on="ds", how="left")

is_grand_total = (F.col("product") == "Total") & (F.col("sales_market") == "Total")
for m in FINAL_MODELS:
    for q in QUANTILES:
        col_name = f"{m}_{q}"
        wide = wide.withColumn(col_name,
            F.when(is_grand_total, F.col(f"gt_{col_name}")).otherwise(F.col(col_name))
        )

# Drop temp columns
for m in FINAL_MODELS:
    for q in QUANTILES:
        wide = wide.drop(f"gt_{m}_{q}")

# Update scenario columns for Grand Total
wide = (
    wide
    .withColumn("worst_case",
        F.when(is_grand_total, F.col("Adaptive_Ensemble_P05")).otherwise(F.col("worst_case")))
    .withColumn("best_case",
        F.when(is_grand_total, F.col("Adaptive_Ensemble_P95")).otherwise(F.col("best_case")))
    .withColumn("Best_Case_Miradore_Adj",
        F.col("best_case") + F.coalesce(F.col("miradore_plan"), F.lit(0)))
    .withColumn("Worst_Case_Miradore_Adj",
        F.col("worst_case") + F.coalesce(F.col("miradore_plan"), F.lit(0)))
)

gt_sample = wide.filter(is_grand_total).select("ds", "best_case", "worst_case").orderBy(F.desc("ds")).limit(3).toPandas()
for _, r in gt_sample.iterrows():
    print(f"  Grand Total {r['ds']}: worst={r['worst_case']:,.0f}, best={r['best_case']:,.0f}, width={r['best_case']-r['worst_case']:,.0f}")

# NOTE on 7-model facade: ~88% of rows are prophet_proxy (pre-Aug 2026) where
# ALL models share identical P50 copied from Prophet. Per-model conformal cannot
# fix this — residuals are identical since P50s are identical. For native V5 rows
# (~12%), models already have DISTINCT bands via different P50s + shared conformal
# shift. The facade percentage decreases naturally as V5 data accumulates daily.

# ============================================================================
# E. RECONCILED POINT FORECAST at market grain
# ============================================================================
# Market-grain raw Adaptive_Ensemble has ~1,915% MAPE. Override ONLY the
# point forecast (most_likely) with top-down reconciled P50. Keep the
# conformal-calibrated best_case/worst_case bands untouched — they have
# ~90% empirical coverage; swapping in reconciled P05/P95 drops coverage
# to ~71% (the reconciled band inherits Total's under-calibrated width).

is_market_level = (F.col("sales_market") != "Total") & (F.col("reconciled_p50").isNotNull())
wide = wide.withColumn("most_likely",
    F.when(is_market_level, F.col("reconciled_p50")).otherwise(F.col("most_likely"))
)
# Re-derive only Most_Likely Miradore adjustment (bands unchanged)
wide = wide.withColumn("Most_Likely_Miradore_Adj",
    F.col("most_likely") + F.coalesce(F.col("miradore_plan"), F.lit(0)))
mkt_overridden = wide.filter(is_market_level).count()
print(f"Section E: Reconciled point forecast applied to {mkt_overridden} market-level rows")
print(f"  (best_case/worst_case retain conformal-calibrated ~90% bands)")

print(f"Final columns: {len(wide.columns)}, rows: {wide.count()}")
"""

In [ ]:
# ============================================================================
# [V2 NO-OP] Metadata + market suppression consolidated into Cell 4.
# ============================================================================
pass
print('V2: Metadata and staging view registered in Cell 4.')
print(f'Staging view row count: {spark.sql("SELECT COUNT(*) FROM _powerbi_wide_staging").collect()[0][0]}')
if False:  # pragma: no cover
    _OLD = """
# METADATA + REGISTER TEMP VIEW FOR SQL WRITE
# ============================================================================

final = (
    wide
    .withColumn("grain_level",
                F.when(F.col("sales_market") == "Total", F.lit("total"))
                 .otherwise(F.lit("market")))
    .withColumn("run_date", F.current_date())
    .withColumn("published_at_utc", F.current_timestamp())
)

# ============================================================================
# 2026-08-31: Market-grain forecast WAPE ~50-62%, deemed unreliable for
# publication (see validation 2026-08-31). Market-grain rows report actuals
# only; forecasts remain at Total/product grain only.
# ============================================================================
_is_market = F.col("grain_level") == "market"

# Null all model quantile + scenario + reconciled columns at market grain.
# NOTE: model_fill_source and is_native_model are intentionally preserved.
# They describe provenance of a forecast that no longer exists for these rows;
# keeping them avoids an extra regression surface and doesn't create wrong
# results (filtering is_native_model=True returns zero market rows, which is
# correct — there is no native forecast at market grain to select).
_SUPPRESS_MODELS = ["Adaptive_Ensemble", "DHR_ARIMA", "ETS", "Global_LGB_Q50", "MSTL_v2", "Prophet"]
_SUPPRESS_Q = ["P05", "P25", "P50", "P75", "P95"]
_null_cols = [f"{m}_{q}" for m in _SUPPRESS_MODELS for q in _SUPPRESS_Q] + [
    "most_likely", "best_case", "worst_case",
    "Best_Case_Miradore_Adj", "Most_Likely_Miradore_Adj", "Worst_Case_Miradore_Adj",
    "reconciled_p50", "reconciled_p05", "reconciled_p95",
]
for _c in _null_cols:
    final = final.withColumn(_c,
        F.when(_is_market, F.lit(None).cast("double")).otherwise(F.col(_c)))

# forecast_type: 'actuals' where actuals non-null, else NULL — market grain only
final = final.withColumn("forecast_type",
    F.when(_is_market & F.col("actuals").isNotNull(), F.lit("actuals"))
     .when(_is_market, F.lit(None).cast("string"))
     .otherwise(F.col("forecast_type")))

# ARR_YTD: NULL past last actuals date per product x market
_last_act = (
    final.filter(_is_market & F.col("actuals").isNotNull())
    .groupBy("product", "sales_market")
    .agg(F.max("ds").alias("_last_act_ds")))
final = final.join(_last_act, on=["product", "sales_market"], how="left")
final = final.withColumn("ARR_YTD",
    F.when(_is_market & (F.col("ds") > F.col("_last_act_ds")), F.lit(None).cast("double"))
     .when(_is_market & F.col("_last_act_ds").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("ARR_YTD")))
final = final.drop("_last_act_ds")

_mkt_n = final.filter(_is_market).count()
_mkt_resid = final.filter(_is_market & F.col("most_likely").isNotNull()).count()
print(f"Market-grain forecast suppression: {_mkt_n} market rows, "
      f"{_mkt_resid} residual (should be 0)")

row_count = final.count()
print(f"Final rows: {row_count}")
print(f"Final columns ({len(final.columns)}):")
for c in sorted(final.columns):
    print(f"  {c}")

# Register as temp view for the SQL write cell
final.createOrReplaceTempView("_powerbi_wide_staging")
print(f"\nRegistered temp view _powerbi_wide_staging")

# --- Sample ---
print("\nSample (5 rows):")
display(final.limit(5))
"""

In [ ]:
# ============================================================================
# FILL MARKET-LEVEL MODEL FORECASTS
# ============================================================================
# Cell 3 Section E nulls ALL model columns at market grain. This cell fills
# them back using:
#   1. ACTUALS rows: proportional allocation of total-level model predictions
#      to each market (so Power BI shows model-vs-actual at market level)
#   2. FORWARD FORECAST rows: create new market rows via trailing 13-week
#      market share allocation from total-level forecasts
#   3. ITSG override: replace proportional values with real market-level
#      model predictions from itsg_forecast_v5
# ============================================================================
import pandas as pd, numpy as np

wide = spark.sql("SELECT * FROM _powerbi_wide_staging").toPandas()
wide['ds'] = pd.to_datetime(wide['ds'])

# 2026-09-30 FIX: a handful of market-grain rows carry a genuine NaN
# sales_market (not the string 'UNKNOWN', which earlier code already mapped
# to 'Unknown') -- this showed up as 394 of 1,928 market rows with no market
# label at all. The upstream SQL in Cell 4 now COALESCEs this at the source,
# but a NaN key could still reach this cell's own trailing_shares groupby
# (pandas has in some versions kept NaN as its own group for object-dtype
# columns), which would then propagate into the proportional-allocation
# forward rows built in Section D below. Belt-and-suspenders: collapse any
# remaining NaN here too, before any grouping happens.
_mkt_rows_mask = wide['grain_level'] == 'market'
wide.loc[_mkt_rows_mask, 'sales_market'] = wide.loc[_mkt_rows_mask, 'sales_market'].fillna('Unknown')

model_cols = [c for c in wide.columns if any(c.startswith(m + '_') for m in MODELS)]
scenario_cols = ['most_likely', 'worst_case', 'best_case', 'blended_p50']
mkt_mask = wide['grain_level'] == 'market'
act_mask = wide['forecast_type'] == 'actuals'

# ── A. Compute trailing 13-week market share from actuals ────────────────
trailing_shares = {}
for prod in ['UCC', 'ITSG']:
    prod_mkt = wide[(wide['product'] == prod) & mkt_mask & act_mask]
    if prod_mkt.empty:
        continue
    max_date = prod_mkt['ds'].max()
    cutoff = max_date - pd.Timedelta(weeks=13)
    recent = prod_mkt[prod_mkt['ds'] > cutoff]
    total_arr = recent['actuals'].sum()
    if total_arr > 0:
        shares = recent.groupby('sales_market')['actuals'].sum() / total_arr
        trailing_shares[prod] = shares.to_dict()
        print(f"{prod} trailing 13w market shares: " +
              ", ".join(f"{k} {v:.1%}" for k, v in sorted(shares.items())))

# ── B. Get total-level model predictions keyed by (product, ds) ──────────
total_rows = wide[wide['grain_level'] == 'total'].copy()
total_lookup = {}
for _, row in total_rows.iterrows():
    total_lookup[(row['product'], row['ds'])] = row

# ── C. Fill existing market ACTUALS rows with allocated model predictions ─
filled_act = 0
for prod in ['UCC', 'ITSG']:
    if prod not in trailing_shares:
        continue
    for mkt_name, share in trailing_shares[prod].items():
        row_mask = (mkt_mask & act_mask &
                    (wide['product'] == prod) &
                    (wide['sales_market'] == mkt_name))
        if row_mask.sum() == 0:
            continue
        for idx in wide.loc[row_mask].index:
            ds_val = wide.loc[idx, 'ds']
            total_row = total_lookup.get((prod, ds_val))
            if total_row is None:
                continue
            for c in model_cols:
                tv = total_row.get(c)
                if pd.notna(tv):
                    wide.loc[idx, c] = tv * share
            # Scenarios: use actual for most_likely, allocated for bands
            wide.loc[idx, 'most_likely'] = wide.loc[idx, 'actuals']
            wide.loc[idx, 'blended_p50'] = wide.loc[idx, 'actuals']
            wc = total_row.get('worst_case')
            bc = total_row.get('best_case')
            wide.loc[idx, 'worst_case'] = wc * share if pd.notna(wc) else np.nan
            wide.loc[idx, 'best_case'] = bc * share if pd.notna(bc) else np.nan
            filled_act += 1
# ── C2. Fallback: when total-level model cols are NULL, fill market rows
#     with actual value (e.g. ITSG Jul-Aug where no hindcast exists) ──────
fallback_filled = 0
still_null = mkt_mask & act_mask & wide[model_cols[0]].isna()
for idx in wide.loc[still_null].index:
    act_val = wide.loc[idx, 'actuals']
    if pd.notna(act_val):
        for c in model_cols:
            wide.loc[idx, c] = act_val
        wide.loc[idx, 'most_likely'] = act_val
        wide.loc[idx, 'blended_p50'] = act_val
        wide.loc[idx, 'worst_case'] = act_val
        wide.loc[idx, 'best_case'] = act_val
        fallback_filled += 1

print(f"\nFilled {filled_act} market actuals rows with allocated model predictions")
print(f"Fallback: filled {fallback_filled} market actuals rows with actual value (no hindcast available)")

# ── D. Create FORWARD market forecast rows (proportional allocation) ─────
total_fwd = wide[(wide['grain_level'] == 'total') &
                 (wide['forecast_type'] == 'forecast') &
                 (wide['product'] != 'Total')]
# 2026-09-30 FIX: this used to create a forecast-type row for every
# (ds, product, market) combination with no check for whether that date had
# already closed with a real market-actuals row sitting in `wide` from an
# EARLIER run's proportional allocation that never got cleaned up once the
# week closed (the forward rows are written every run, but nothing retired
# the stale one from a prior run once the real actuals row for that same
# key showed up). Confirmed: 18 duplicate (ds, product, sales_market) rows,
# all on a single already-closed date, each with one real actuals row and
# one leftover forecast row. Fix: skip creating/keeping a forward row for
# any key that already has a real actuals row in `wide`.
_existing_actuals_keys = set(
    zip(wide.loc[mkt_mask & act_mask, 'product'],
        wide.loc[mkt_mask & act_mask, 'sales_market'],
        wide.loc[mkt_mask & act_mask, 'ds'])
)
new_mkt_rows = []
_skipped_stale = 0
for _, row in total_fwd.iterrows():
    prod = row['product']
    if prod not in trailing_shares:
        continue
    for mkt_name, share in trailing_shares[prod].items():
        if (prod, mkt_name, row['ds']) in _existing_actuals_keys:
            # This week already closed with real market actuals (from a
            # prior run) -- don't add a duplicate forward-forecast row.
            _skipped_stale += 1
            continue
        new_row = row.copy()
        new_row['sales_market'] = mkt_name
        new_row['grain_level'] = 'market'
        new_row['model_fill_source'] = 'proportional_allocation'
        for c in model_cols:
            if pd.notna(row[c]):
                new_row[c] = row[c] * share
        for c in scenario_cols:
            if c in row and pd.notna(row[c]):
                new_row[c] = row[c] * share
        new_row['actuals'] = np.nan
        new_row['ARR_YTD'] = np.nan
        new_row['mc_actuals'] = np.nan
        new_mkt_rows.append(new_row)

# Also drop any STALE forward row already sitting in `wide` from a prior run
# for a key that now has real actuals (same bug, but for rows that were
# already written to the table before this fix existed).
_stale_fwd_mask = (
    mkt_mask & (wide['forecast_type'] == 'forecast') &
    wide.apply(lambda r: (r['product'], r['sales_market'], r['ds']) in _existing_actuals_keys, axis=1)
)
_n_stale_removed = int(_stale_fwd_mask.sum())
if _n_stale_removed:
    wide = wide.loc[~_stale_fwd_mask].reset_index(drop=True)
    mkt_mask = wide['grain_level'] == 'market'
    act_mask = wide['forecast_type'] == 'actuals'

if new_mkt_rows:
    mkt_fwd_df = pd.DataFrame(new_mkt_rows)
    wide = pd.concat([wide, mkt_fwd_df], ignore_index=True)
    wide = wide.sort_values(['product', 'sales_market', 'ds']).reset_index(drop=True)
    mkt_mask = wide['grain_level'] == 'market'
    act_mask = wide['forecast_type'] == 'actuals'
    print(f"Added {len(new_mkt_rows)} forward market forecast rows "
          f"({_skipped_stale} skipped -- already had real actuals; "
          f"{_n_stale_removed} stale forward rows from a prior run removed)")

# ── E. ITSG override: real market-level model predictions ────────────────
try:
    itsg_mkt_v5 = spark.sql(f"""
        SELECT CAST(ds AS TIMESTAMP) AS ds, sales_market, model,
               p05, p25, p50, p75, p95
        FROM {CATALOG}.{SCHEMA}.itsg_forecast_v5
        WHERE grain_level = 'market'
    """).toPandas()
    itsg_mkt_v5['ds'] = pd.to_datetime(itsg_mkt_v5['ds'])
    itsg_mkt_v5['model_name'] = itsg_mkt_v5['model'].replace(MODEL_RENAME)

    _q_map_v5 = {'P05': 'p05', 'P25': 'p25', 'P50': 'p50', 'P75': 'p75', 'P95': 'p95'}
    overrides = 0
    for (ds_val, mkt_name), grp in itsg_mkt_v5.groupby(['ds', 'sales_market']):
        row_mask = ((wide['product'] == 'ITSG') & (wide['ds'] == ds_val) &
                    (wide['sales_market'] == mkt_name) & (wide['grain_level'] == 'market'))
        if row_mask.sum() == 0:
            continue
        for _, mr in grp.iterrows():
            m = mr['model_name']
            if m not in MODELS:
                continue
            for q_label, q_col in _q_map_v5.items():
                col_name = f'{m}_{q_label}'
                if col_name in wide.columns:
                    wide.loc[row_mask, col_name] = mr[q_col]
            overrides += 1
    print(f"ITSG: overrode {overrides} market model cells with itsg_forecast_v5 data")
except Exception as e:
    print(f"ITSG market override skipped: {e}")

# ── F. Recompute ARR_YTD for market rows ─────────────────────────────────
mkt_mask = wide['grain_level'] == 'market'
for prod in wide.loc[mkt_mask, 'product'].unique():
    for mkt in wide.loc[mkt_mask & (wide['product'] == prod), 'sales_market'].unique():
        pm = mkt_mask & (wide['product'] == prod) & (wide['sales_market'] == mkt)
        act_vals = wide.loc[pm, 'actuals']
        ytd = act_vals.groupby(wide.loc[pm, 'year']).cumsum()
        last_act_ds = wide.loc[pm & wide['actuals'].notna(), 'ds']
        if not last_act_ds.empty:
            ytd[wide.loc[pm, 'ds'] > last_act_ds.max()] = np.nan
        wide.loc[pm, 'ARR_YTD'] = ytd

# ── G. Re-register staging view ──────────────────────────────────────────
sdf = spark.createDataFrame(wide)
sdf.createOrReplaceTempView('_powerbi_wide_staging')

mkt_total = (wide['grain_level'] == 'market').sum()
mkt_with_model = mkt_total - wide.loc[wide['grain_level'] == 'market', model_cols[0]].isna().sum()
print(f"\n✓ _powerbi_wide_staging refreshed: {len(wide)} rows")
print(f"  Market rows: {mkt_total} ({mkt_with_model} with model data)")
print(f"  Total rows:  {(wide['grain_level']=='total').sum()}")

In [ ]:
# ============================================================================
# PERIOD-END UPLIFT: DIAGNOSTIC ONLY as of 2026-10-05 (was a live multiplier)
# ============================================================================
# FIX (2026-10-05): this cell's "year(ds) < year(current_date)" lookback filter
# can never match any row -- _powerbi_wide_staging is deliberately scoped to
# start ~Jan 2026 and run ~1 year forward (established earlier in this
# project), so there is no row with a year strictly before the current one.
# Year-end and Quarter-end were therefore permanently stuck at the 1.0x
# fallback, silently providing zero correction while looking active.
#
# Widening the lookback to use ANY past closed period-end week (not just
# prior calendar years) was considered and rejected: UCC's own Step 14
# QE_CALIBRATION already computes and applies a quarter-end correction
# upstream (1.236x as of 2026-10-05, derived from UCC's own holdout/backtest
# evidence, with its own double-counting guard against UCC's monthly
# calibration -- see that notebook's 2026-10-05 fix). This cell's ratio
# (actuals / Adaptive_Ensemble_P50) would be measured on the SAME 2026
# quarter-end weeks UCC already used to calibrate itself, so activating this
# as a SECOND live multiplier risks re-stacking a correction on top of a
# correction -- the exact bug class just fixed in UCC today, one layer
# downstream and harder to see.
#
# Kept as a DIAGNOSTIC: still computes and PRINTS what each factor would be
# (so a residual gap downstream of UCC's own calibration stays visible), but
# no longer multiplies it into the output columns. UPLIFT_* is left at 1.0
# (no-op) everywhere below; only the diagnostic print reports the live-measured
# ratio. Re-enable as a live correction only with a deliberate decision to
# retire or coordinate with UCC's own QE_CALIBRATION, not as a silent default.
# ============================================================================
# V2: LIVE-COMPUTED bias factor (was: frozen constants QE_UPLIFT=1.5 / ME_UPLIFT=1.3
# based on a one-time historical measurement: "+18.7% under-forecast bias").
#
# 2026-09-29: split into THREE tiers, not two. Fiscal year-end (Q4, since
# GoTo FY = calendar year) is pulled out from ordinary quarter-end. The
# budget-flush effect at FY close (unspent budget doesn't roll over) is a
# materially bigger, separately-documented pattern than a routine Q1/Q2/Q3
# close, so collapsing both into one "quarter-end" factor would understate
# FY-end and/or overstate a normal quarter-end. Precedence: year-end >
# quarter-end > month-end (a week is exactly one of the three).
#
# Why this changed: the upstream UCC/ITSG models now include explicit
# quarter-end features (quarter_end_flag, fy_end_proximity, etc.) built
# specifically to capture end-of-period deal acceleration. A frozen
# multiplier on top of that risks double-counting a correction the models
# already make, and a number measured once will silently drift stale as
# model behavior changes. So instead we recompute the bias fresh each run
# from recent CLOSED weeks (actuals IS NOT NULL) in _powerbi_wide_staging,
# comparing actuals to Adaptive_Ensemble_P50 (the model's own point-forecast
# quantile column — NOT `most_likely`, which is already overwritten with the
# actual value for closed weeks and would make the ratio circular).
#
# Method:
#   ratio_week = actuals / Adaptive_Ensemble_P50   (Total grain, closed weeks only)
#   YE_UPLIFT  = median(ratio_week) over closed fiscal year-end weeks, PRIOR YEARS only
#   QE_UPLIFT  = median(ratio_week) over closed quarter-end (non-YE) weeks, PRIOR YEARS only
#   ME_UPLIFT  = median(ratio_week) over closed month-end (non-quarter-end) weeks, PRIOR YEARS only
#   Lookback window: ALL closed history from years strictly before the current
#     calendar year (2026-09-30: widened from a rolling 8-quarter/730-day
#     window to match forecast_unify_publish's forecast_app_snapshot uplift
#     cell, which uses the same "all prior years" definition -- both
#     consumers now derive uplift the same way. A fixed 730-day window was
#     also too thin for the year-end tier specifically: at most one FY-end
#     week per year falls in any 8-quarter window, so it took years to clear
#     the minimum sample size; using all available prior years gets there
#     faster without changing the method).
#   Floor at 1.0 (never shrinks the forecast — only lifts genuine under-forecasts).
#   Cap at 3.0 (a single noisy week can't blow up the multiplier).
#   Minimum sample size: 4 closed weeks per tier; below that, fall back to the
#     documented default of 1.0 (NO uplift) rather than silently reusing the
#     old 1.5x/1.3x, since those are exactly the numbers now considered
#     unverified/possibly stale.
# Only affects forecast rows at Total grain — actuals untouched.
# ============================================================================
from pyspark.sql import functions as F

UPLIFT_MIN_SAMPLE    = 4         # minimum closed weeks required to trust the ratio
UPLIFT_FLOOR         = 1.0       # never shrink the forecast
UPLIFT_CAP           = 3.0       # guard against a single noisy data point
UPLIFT_DEFAULT       = 1.0       # fallback when there isn't enough stable history

wide_adj = spark.table('_powerbi_wide_staging')

# Detect period-end weeks:
#   Year-end    = next week falls in a different calendar year (FY = calendar year)
#   Quarter-end = next week falls in a different quarter, excluding year-end
#   Month-end   = next week falls in a different month, excluding quarter-end
wide_adj = (wide_adj
    .withColumn('_next_month', F.month(F.date_add(F.col('ds'), 7)))
    .withColumn('_next_quarter', F.quarter(F.date_add(F.col('ds'), 7)))
    .withColumn('_next_year', F.year(F.date_add(F.col('ds'), 7)))
    .withColumn('_is_total', F.col('grain_level') == 'total')
    .withColumn('_is_forecast_total',
        (F.col('forecast_type') == 'forecast') & F.col('_is_total'))
    .withColumn('_is_ye_week',
        F.col('_is_total') & (F.year(F.col('ds')) != F.col('_next_year')))
    .withColumn('_is_qe_week',
        F.col('_is_total') & (F.quarter(F.col('ds')) != F.col('_next_quarter')) & ~F.col('_is_ye_week'))
    .withColumn('_is_me_week',
        F.col('_is_total') & (F.month(F.col('ds')) != F.col('_next_month'))
        & ~F.col('_is_qe_week') & ~F.col('_is_ye_week'))
    .withColumn('_is_ye', F.col('_is_forecast_total') & F.col('_is_ye_week'))
    .withColumn('_is_qe', F.col('_is_forecast_total') & F.col('_is_qe_week'))
    .withColumn('_is_me', F.col('_is_forecast_total') & F.col('_is_me_week'))
)

# ----------------------------------------------------------------------------
# LIVE BIAS COMPUTATION: actual / Adaptive_Ensemble_P50 over recent CLOSED
# period-end weeks (Total grain only).
# ----------------------------------------------------------------------------
_max_closed_ds = (wide_adj
    .filter(F.col('_is_total') & (F.col('actuals').isNotNull()))
    .agg(F.max('ds')).collect()[0][0])

def _compute_live_uplift(week_flag_col, label):
    if _max_closed_ds is None:
        return UPLIFT_DEFAULT, 0
    _closed = (wide_adj
        .filter(
            F.col('_is_total')
            & F.col(week_flag_col)
            & F.col('actuals').isNotNull()
            & F.col('Adaptive_Ensemble_P50').isNotNull()
            & (F.col('Adaptive_Ensemble_P50') != 0)
            & (F.year(F.col('ds')) < F.year(F.current_date()))
            & (F.col('ds') <= F.lit(_max_closed_ds))
        )
        .withColumn('_ratio', F.col('actuals') / F.col('Adaptive_Ensemble_P50'))
    )
    _n = _closed.count()
    if _n < UPLIFT_MIN_SAMPLE:
        print(f'  {label}: only {_n} closed weeks in lookback (< min {UPLIFT_MIN_SAMPLE}) '
              f'- falling back to default {UPLIFT_DEFAULT}x (no uplift)')
        return UPLIFT_DEFAULT, _n
    _median_ratio = _closed.approxQuantile('_ratio', [0.5], 0.001)[0]
    _factor = max(UPLIFT_FLOOR, min(UPLIFT_CAP, _median_ratio))
    print(f'  {label}: {_n} closed weeks in lookback, median actual/forecast ratio = '
          f'{_median_ratio:.3f}x -> live factor = {_factor:.3f}x')
    return _factor, _n

print('Computing live period-end bias factors '
      f'(lookback=all prior calendar years, min_sample={UPLIFT_MIN_SAMPLE}, '
      f'floor={UPLIFT_FLOOR}x, cap={UPLIFT_CAP}x):')
YE_UPLIFT, _ye_sample_n = _compute_live_uplift('_is_ye_week', 'Year-end')
QE_UPLIFT, _qe_sample_n = _compute_live_uplift('_is_qe_week', 'Quarter-end')
ME_UPLIFT, _me_sample_n = _compute_live_uplift('_is_me_week', 'Month-end')

# DIAGNOSTIC ONLY (2026-10-05): report what each factor WOULD be and how many
# rows it WOULD touch, but do not apply it -- see header note above for why.
_model_q_cols = [c for c in wide_adj.columns
                 if any(c.startswith(m + '_P') for m in MODELS)]
_scenario_cols = ['most_likely', 'worst_case', 'best_case']

_ye_count = wide_adj.filter(F.col('_is_ye')).count()
_qe_count = wide_adj.filter(F.col('_is_qe')).count()
_me_count = wide_adj.filter(F.col('_is_me')).count()
print('Period-end uplift -- DIAGNOSTIC ONLY, NOT applied to output '
      f'(all prior years, floor={UPLIFT_FLOOR}x, cap={UPLIFT_CAP}x):')
print(f'  Year-end    would be {YE_UPLIFT:.3f}x, from {_ye_sample_n} closed weeks ({_ye_count} forecast rows would be touched)')
print(f'  Quarter-end would be {QE_UPLIFT:.3f}x, from {_qe_sample_n} closed weeks ({_qe_count} forecast rows would be touched)')
print(f'  Month-end   would be {ME_UPLIFT:.3f}x, from {_me_sample_n} closed weeks ({_me_count} forecast rows would be touched)')
print('  (Not applied: UCC/ITSG already apply their own quarter-end calibration '
      'upstream; multiplying again here would risk double-counting. See header note.)')

for r in (wide_adj
    .filter(F.col('_is_ye') | F.col('_is_qe') | F.col('_is_me'))
    .select('ds', 'product',
        F.when(F.col('_is_ye'), F.lit('year-end'))
         .when(F.col('_is_qe'), F.lit('quarter-end'))
         .otherwise(F.lit('month-end')).alias('type'),
        'most_likely')
    .orderBy('ds', 'product')
    .limit(20).collect()):
    print(f"  {r['ds']}  {r['product']:>6s}  {r['type']:<12s}  most_likely=${r['most_likely']:>12,.0f}")

# Cleanup and re-register staging view
wide_adj = wide_adj.drop('_next_month', '_next_quarter', '_next_year', '_is_total',
                          '_is_forecast_total', '_is_ye_week', '_is_qe_week', '_is_me_week',
                          '_is_ye', '_is_qe', '_is_me')
wide_adj.createOrReplaceTempView('_powerbi_wide_staging')
wide = wide_adj  # Update reference for downstream cells
print(f'✓ Re-registered _powerbi_wide_staging with period-end adjustments')


In [ ]:
# ============================================================================
# TOTAL GRAIN ONLY  (2026-10-08)
# ============================================================================
# The published tables carry only Total, UCC and ITSG at total grain -- no
# region/market rows. Market-grain rows were allocated shares of the total, had
# no Static/Dynamic values on closed weeks, and for ITSG did not add back up to
# the total (differences of $1.6K-$12.8K per week). Filtering here (before the
# long-format table is built and before any write) means the wide table, the
# long table, the history table and the federated copy are all total-only.
# NOTE: anything that filters this table by sales_market other than 'Total'
# (e.g. a region selector reading arr_forecast_powerbi_wide) will now get no rows.
from pyspark.sql import functions as F

_all = spark.table('_powerbi_wide_staging')
_before = _all.count()
_total_only = _all.filter(F.col('grain_level') == 'total')
_total_only.createOrReplaceTempView('_powerbi_wide_staging')
wide = _total_only
_after = spark.table('_powerbi_wide_staging').count()

_products = sorted(r['product'] for r in spark.table('_powerbi_wide_staging').select('product').distinct().collect())
assert set(_products) <= {'UCC', 'ITSG', 'Total'}, f'unexpected products after total-only filter: {_products}'
assert _after > 0, 'total-only filter removed every row -- check grain_level values in staging'
_mkt = spark.table('_powerbi_wide_staging').filter(F.col('sales_market') != 'Total').count()
assert _mkt == 0, f'{_mkt} non-Total sales_market rows remain at total grain'

_sd = (spark.table('_powerbi_wide_staging').filter(F.col('ds') >= F.lit('2026-01-01'))
       .agg(F.count('*').alias('rows'),
            F.sum(F.col('Static_P50').isNull().cast('int')).alias('static_null'),
            F.sum(F.col('Dynamic_P50').isNull().cast('int')).alias('dynamic_null')).collect()[0])
print(f"Total-grain only: kept {_after:,} of {_before:,} staging rows; products={_products}")
print(f"2026+ rows={_sd['rows']:,} | Static_P50 nulls={_sd['static_null']} | Dynamic_P50 nulls={_sd['dynamic_null']}")


In [ ]:
# ============================================================================
# LONG-FORMAT FACT TABLE  (V2 — sourced from _powerbi_wide_staging)
# ============================================================================
# Unpivots wide staging view into:
#   ds | product | sales_market | model | quantile | forecast_value
# Plus shared columns (actuals, mc_actuals, etc.)
# ============================================================================
from pyspark.sql import functions as F
from functools import reduce

staging = spark.table('_powerbi_wide_staging')
staging_cols = staging.columns  # Cache to avoid repeated Analyze RPCs

LONG_MODELS = MODELS  # Same 6 models from Config

# Build union of model × quantile stacks
stacks = []
for m in LONG_MODELS:
    for q in QUANTILES:
        col = f"{m}_{q}"
        if col in staging_cols:
            stacks.append(
                staging.select(
                    "ds", "product", "sales_market", "grain_level",
                    "forecast_type", "model_fill_source",
                    "year", "quarter", "month", "ISOWeek",
                    F.lit(m).alias("model"),
                    F.lit(q).alias("quantile"),
                    F.col(col).alias("forecast_value"),
                    # V2: all forecasts are genuine (no proxy backfill)
                    F.lit(True).alias("is_distinct_forecast"),
                )
            )

long_forecasts = reduce(lambda a, b: a.unionByName(b), stacks)

# Per-model holdout WAPE from monitoring baseline
_model_map = spark.createDataFrame([
    ("Adaptive_Ensemble", "Adaptive_Ensemble"),
    ("DHR_ARIMA", "DHR_ARIMA"),
    ("ETS", "ETS"),
    ("Global_LGB_Q50", "Global_LGB_Q50_UCC"),
    ("MSTL_v2", "MSTL_v2"),
    ("Prophet", "Prophet_trend"),
], ["model", "bl_model"])

try:
    _ucc_baseline = spark.table(f"{CATALOG}.{SCHEMA}.ucc_forecast_monitoring_baseline")
    _latest_bl_date = _ucc_baseline.agg(F.max("run_date_utc")).collect()[0][0]
    _bl_wape = (
        _ucc_baseline.filter(F.col("run_date_utc") == _latest_bl_date)
        .select(F.col("model").alias("bl_model"),
                F.round("holdout_wape_baseline", 1).alias("model_holdout_wape"))
        .dropDuplicates(["bl_model"])
    )
    long_forecasts = (
        long_forecasts
        .join(_model_map, on="model", how="left")
        .join(_bl_wape, on="bl_model", how="left")
        .drop("bl_model")
    )
    print(f"Per-model holdout WAPE joined ({_bl_wape.count()} models from baseline {_latest_bl_date})")
except Exception as e:
    print(f"Monitoring baseline not available, skipping WAPE join: {e}")
    long_forecasts = long_forecasts.withColumn("model_holdout_wape", F.lit(None).cast("double"))

# Add shared columns
shared_cols = (
    staging.select(
        "ds", "product", "sales_market",
        "actuals", "mc_actuals", "miradore_plan",
        "run_date", "published_at_utc",
    )
    .distinct()
)

long_fact = long_forecasts.join(shared_cols, on=["ds", "product", "sales_market"], how="left")

# Filter: only rows with actual forecast values (drop market-grain nulls)
pre_filter = long_fact.count()
long_fact = long_fact.filter(F.col("forecast_value").isNotNull())
long_count = long_fact.count()
print(f"Long-format rows: {long_count:,} (dropped {pre_filter - long_count:,} null-forecast rows)")
print(f"Long-format columns: {len(long_fact.columns)}")

long_fact.createOrReplaceTempView("_long_format_staging")
print(f"\u2713 Registered _long_format_staging")
print(f"\nSample:")
display(long_fact.limit(5))

In [ ]:
%sql
-- Write both wide and long tables to Unity Catalog (latest snapshot)
CREATE OR REPLACE TABLE datagroup_mdl.mdl_sales_analytics.arr_forecast_powerbi_wide AS
SELECT * FROM _powerbi_wide_staging;

CREATE OR REPLACE TABLE datagroup_mdl.mdl_sales_analytics.arr_forecast_long AS
SELECT * FROM _long_format_staging;

-- Run history: DROP + recreate to absorb schema changes (LGB column merge,
-- new accuracy_eligible / ensemble_holdout_wape columns).
-- Remove the DROP once schema is stable; CREATE IF NOT EXISTS will then be a no-op.
DROP TABLE IF EXISTS datagroup_mdl.mdl_sales_analytics.arr_forecast_wide_history;

CREATE TABLE IF NOT EXISTS datagroup_mdl.mdl_sales_analytics.arr_forecast_wide_history
AS SELECT * FROM datagroup_mdl.mdl_sales_analytics.arr_forecast_powerbi_wide WHERE 1=0;

-- Dedup: remove any prior rows for today's run_date before re-appending,
-- so repeated notebook runs don't stack duplicate snapshots.
DELETE FROM datagroup_mdl.mdl_sales_analytics.arr_forecast_wide_history
WHERE run_date IN (SELECT DISTINCT run_date FROM _powerbi_wide_staging);

INSERT INTO datagroup_mdl.mdl_sales_analytics.arr_forecast_wide_history
SELECT * FROM _powerbi_wide_staging;

-- Also write to federated.sales for cross-catalog app access
CREATE OR REPLACE TABLE federated.sales.arr_forecast_powerbi_wide AS
SELECT * FROM _powerbi_wide_staging

In [ ]:
import base64

# Read table and convert to pandas
df = spark.table(OUTPUT_TABLE).toPandas()
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")

# Create downloadable CSV link
csv_data = df.to_csv(index=False)
b64 = base64.b64encode(csv_data.encode()).decode()
filename = "arr_forecast_powerbi_wide.csv"
html = f'<a download="{filename}" href="data:text/csv;base64,{b64}" style="font-size:16px; padding:10px 20px; background:#1B3A4B; color:white; text-decoration:none; border-radius:5px;">Download {filename} ({len(df)} rows, {len(df.columns)} cols)</a>'
displayHTML(html)

In [ ]:
# ============================================================================
# GRANTS
# ============================================================================

LONG_TABLE = f"{CATALOG}.{SCHEMA}.arr_forecast_long"
HISTORY_TABLE = f"{CATALOG}.{SCHEMA}.arr_forecast_wide_history"
spark.sql(f"GRANT SELECT ON TABLE {OUTPUT_TABLE} TO `{APP_SERVICE_PRINCIPAL}`")
spark.sql(f"GRANT SELECT ON TABLE {LONG_TABLE} TO `{APP_SERVICE_PRINCIPAL}`")
spark.sql(f"GRANT SELECT ON TABLE {HISTORY_TABLE} TO `{APP_SERVICE_PRINCIPAL}`")
print(f"Granted SELECT on {OUTPUT_TABLE} to {APP_SERVICE_PRINCIPAL}")
print(f"Granted SELECT on {LONG_TABLE} to {APP_SERVICE_PRINCIPAL}")
print(f"Granted SELECT on {HISTORY_TABLE} to {APP_SERVICE_PRINCIPAL}")

# Federated sales grants — full chain: catalog → schema → table
FED_CAT    = "federated"
FED_SCHEMA = "federated.sales"
FED_WIDE   = "federated.sales.arr_forecast_powerbi_wide"
try:
    spark.sql(f"GRANT USE CATALOG ON CATALOG {FED_CAT} TO `{APP_SERVICE_PRINCIPAL}`")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {FED_SCHEMA} TO `{APP_SERVICE_PRINCIPAL}`")
    spark.sql(f"GRANT SELECT ON TABLE {FED_WIDE} TO `{APP_SERVICE_PRINCIPAL}`")
    print(f"Granted full access chain on {FED_WIDE} to {APP_SERVICE_PRINCIPAL}")
except Exception as e:
    print(f"⚠ federated.sales grant skipped: {e}")

In [ ]:
# ============================================================================
# FEDERATED SALES SYNC — Copy all app-critical tables to federated.sales
# ============================================================================
# This cell runs at the END of the pipeline (last task). All upstream tables
# have been written by UCC, ITSG, Unify, Hindcast, and PowerBI notebooks.
# We read from datagroup_mdl and write to federated.sales so the app can
# switch to federated.sales with zero pipeline changes.
#
# forecast_app_snapshot  — already written in forecast_unify_publish
# arr_forecast_powerbi_wide — already written in Cell 6b above
# This cell handles the remaining 14 tables.
#
# COST FIX: ucc_forecast_vintages / itsg_forecast_vintages are append-only
# backtest tables in the SOURCE catalog — they accumulate one new snapshot's
# worth of rows per day, forever (see the UCC/ITSG notebooks' own design).
# A full overwrite copy of these two tables every day would re-transfer the
# entire, ever-growing history each run, so the cost of this sync step would
# grow without bound over the life of the pipeline. These two tables are
# synced INCREMENTALLY instead (initial full copy once, then append-only new
# rows by `run_date_utc`, the per-run stamp already used for these vintages /
# baseline tables elsewhere in this pipeline — see Cell 10's use of
# `run_date_utc` against `ucc_forecast_monitoring_baseline`, the sibling
# per-run-stamped table). All other tables below are bounded / current-state
# tables where a full daily refresh is cheap and appropriate, so they are
# left on the original full-overwrite path.
# ============================================================================

from pyspark.sql import functions as F

SRC = f"{CATALOG}.{SCHEMA}"   # datagroup_mdl.mdl_sales_analytics
FED = "federated.sales"
SP  = APP_SERVICE_PRINCIPAL

# Tables synced via full overwrite every run (bounded / current-state tables)
FED_TABLES = [
    # Leaderboard + Accuracy
    "forecast_leaderboard_v2compat",   # VIEW → materialized
    "forecast_quarterly_accuracy",
    # AI Insights
    "forecast_insights",
    "forecast_insights_v2compat",      # VIEW → materialized
    "forecast_graph_explanations",
    # Exec Mode
    "forecast_scenario_grid",
    "forecast_sensitivity",
    "forecast_model_scenarios",
    "forecast_historical_scenarios",
    # Backtest / Accuracy
    "ucc_forecast_cbe",
    "itsg_forecast_cbe",
    # Legacy (Model Lab — being replaced)
    "arr_forecast_app_latest",
]

# Tables synced incrementally (append-only, ever-growing vintages tables)
FED_VINTAGES_TABLES = [
    "ucc_forecast_vintages",
    "itsg_forecast_vintages",
]
VINTAGE_STAMP_COL = "run_date_utc"  # per-run stamp on vintage rows (see Cell 10)

print(f"Syncing {len(FED_TABLES) + len(FED_VINTAGES_TABLES)} tables to {FED}...\n")

success = 0
skipped = 0

# --- Full-overwrite tables (unchanged behavior) ---
for tbl in FED_TABLES:
    src_fqn = f"{SRC}.{tbl}"
    fed_fqn = f"{FED}.{tbl}"
    try:
        df = spark.table(src_fqn)
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fed_fqn)
        spark.sql(f"GRANT SELECT ON TABLE {fed_fqn} TO `{SP}`")
        cnt = df.count()
        success += 1
        print(f"  ✓ {tbl} ({cnt:,} rows)")
    except Exception as e:
        skipped += 1
        print(f"  ⚠ {tbl}: {str(e)[:120]}")

# --- Incremental append-only sync for the ever-growing vintages tables ---
for tbl in FED_VINTAGES_TABLES:
    src_fqn = f"{SRC}.{tbl}"
    fed_fqn = f"{FED}.{tbl}"
    try:
        fed_exists = False
        try:
            fed_exists = spark.catalog.tableExists(fed_fqn)
        except Exception as e_exists:
            print(f"  ⚠ {tbl}: could not check federated table existence ({str(e_exists)[:100]}), assuming not present")
            fed_exists = False

        if not fed_exists:
            # First-ever sync: seed federated table with full history once.
            df = spark.table(src_fqn)
            df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fed_fqn)
            spark.sql(f"GRANT SELECT ON TABLE {fed_fqn} TO `{SP}`")
            cnt = df.count()
            success += 1
            print(f"  ✓ {tbl} ({cnt:,} rows) [initial full seed]")
        else:
            # Incremental: only append rows newer than what's already federated.
            max_synced = (
                spark.table(fed_fqn)
                .agg(F.max(VINTAGE_STAMP_COL).alias("m"))
                .collect()[0]["m"]
            )
            src_df = spark.table(src_fqn)

            if max_synced is None:
                # Federated table exists but is empty — treat as initial seed.
                new_rows_df = src_df
            else:
                # Idempotency guard: source writes are already partition-scoped
                # per run, so a same-day rerun normally yields zero new rows
                # under a strict `>` comparison. As an extra guard against a
                # same-day rerun landing different row content under the same
                # run_date_utc, delete any previously-synced rows for that
                # exact stamp before appending, scoped only to that value.
                same_stamp_fed = spark.table(fed_fqn).filter(
                    F.col(VINTAGE_STAMP_COL) == F.lit(max_synced)
                )
                if same_stamp_fed.limit(1).count() > 0:
                    spark.sql(
                        f"DELETE FROM {fed_fqn} WHERE {VINTAGE_STAMP_COL} = "
                        f"(SELECT MAX({VINTAGE_STAMP_COL}) FROM (SELECT {VINTAGE_STAMP_COL} FROM {fed_fqn}))"
                    )

                new_rows_df = src_df.filter(F.col(VINTAGE_STAMP_COL) > F.lit(max_synced))

            new_cnt = new_rows_df.count()
            if new_cnt > 0:
                new_rows_df.write.format("delta").mode("append").saveAsTable(fed_fqn)
                spark.sql(f"GRANT SELECT ON TABLE {fed_fqn} TO `{SP}`")
                success += 1
                print(f"  ✓ {tbl}: appended {new_cnt:,} new rows (skipped re-sync of unchanged history)")
            else:
                spark.sql(f"GRANT SELECT ON TABLE {fed_fqn} TO `{SP}`")
                success += 1
                print(f"  ✓ {tbl}: 0 new rows since last sync (up to date, no re-sync needed)")
    except Exception as e:
        skipped += 1
        print(f"  ⚠ {tbl}: {str(e)[:120]}")

total_tables = len(FED_TABLES) + len(FED_VINTAGES_TABLES)
print(f"\nFederated sync complete: {success}/{total_tables} tables written, {skipped} skipped")
if success > 0:
    print(f"All granted to SP {SP}")
